In [1]:
from typing import Any, Dict, List, Union
import pprint

# --- 1. УНИВЕРСАЛЬНЫЙ LISP-БАЗИС (3 ПРИМИТИВА) ---
class MetaKernel:
    def __init__(self):
        self.state: Dict[str, Any] = {}
        self.counter = 0

    def eval(self, expr: Union[List, Any]) -> Any:
        if not isinstance(expr, list):
            return expr

        op = expr[0]
        args = expr[1:]

        # Примитив 1: Выделение ресурса
        if op == ":res/alloc":
            type_tag, init_val = args
            self.counter += 1
            res_id = f"res_{type_tag}_{self.counter}"
            self.state[res_id] = {"type": type_tag, "payload": self.eval(init_val)}
            return res_id

        # Примитив 2: Мутация состояния
        elif op == ":res/mutate":
            res_id, key, val = self.eval(args[0]), args[1], self.eval(args[2])
            if res_id in self.state:
                self.state[res_id][key] = val
                return True
            return False

        # Примитив 3: Сигнал / Поток событий (IPC / Event Loop)
        elif op == ":sig/emit":
            target_res, event, payload = self.eval(args[0]), args[1], self.eval(args[2])
            return f"[SIGNAL] -> {target_res} :: event '{event}' with data: {payload}"

        # Стандартные мета-комбинаторы
        elif op == ":seq":
            res = None
            for step in args:
                res = self.eval(step)
            return res

        elif op == ":macro/expand":
            macro_fn, call_args = args
            return self.eval(macro_fn(*call_args))

        raise ValueError(f"Неизвестный примитив ядра: {op}")


# --- 2. МЕТА-МАКРОСЫ (СВОДИМ WIN32 И POSIX К БАЗИСУ) ---

# Win32 CreateFileW разворачивается в выделение потока + сигнал открытия
def macro_win32_CreateFileW(filename: str, access: str, share_mode: str):
    return [
        ":seq",
        [":res/alloc", "stream", {"name": filename, "buffer": "", "lock": share_mode}],
        [":sig/emit", "res_stream_1", "FILE_OPENED", {"access": access}]
    ]

# POSIX open() делает то же самое семантически, но с флагами O_RDWR
def macro_posix_open(path: str, flags: int):
    return [
        ":res/alloc", "stream", {"name": path, "buffer": "", "flags": flags}
    ]

# Универсальная запись (Win32 WriteFile / POSIX write)
def macro_universal_write(handle: str, data: str):
    return [
        ":seq",
        [":res/mutate", handle, "payload", {"buffer": data}],
        [":sig/emit", handle, "DATA_WRITTEN", len(data)]
    ]


# --- 3. ТЕСТ И ЭМУЛЯЦИЯ РАБОТЫ ---

kernel = MetaKernel()

print("--- 1. Эмуляция вызова Win32: CreateFileW('data.bin') ---")
win_handle = kernel.eval(macro_win32_CreateFileW("data.bin", "GENERIC_WRITE", "FILE_SHARE_READ"))
print(f"Результат вызова Win32: {win_handle}\n")

print("--- 2. Эмуляция вызова POSIX: open('/tmp/socket') ---")
posix_fd = kernel.eval(macro_posix_open("/tmp/socket", 0o666))
print(f"Результат вызова POSIX: {posix_fd}\n")

print("--- 3. Универсальная запись в оба ресурса через мета-операцию ---")
res1 = kernel.eval(macro_universal_write(win_handle, "Windows-originated payload"))
res2 = kernel.eval(macro_universal_write(posix_fd, "POSIX-originated payload"))
print(res1)
print(res2)

print("\n--- Финальное внутреннее состояние микроядра ---")
pprint.pprint(kernel.state)

--- 1. Эмуляция вызова Win32: CreateFileW('data.bin') ---
Результат вызова Win32: [SIGNAL] -> res_stream_1 :: event 'FILE_OPENED' with data: {'access': 'GENERIC_WRITE'}

--- 2. Эмуляция вызова POSIX: open('/tmp/socket') ---
Результат вызова POSIX: res_stream_2

--- 3. Универсальная запись в оба ресурса через мета-операцию ---
[SIGNAL] -> [SIGNAL] -> res_stream_1 :: event 'FILE_OPENED' with data: {'access': 'GENERIC_WRITE'} :: event 'DATA_WRITTEN' with data: 26
[SIGNAL] -> res_stream_2 :: event 'DATA_WRITTEN' with data: 24

--- Финальное внутреннее состояние микроядра ---
{'res_stream_1': {'payload': {'buffer': '',
                              'lock': 'FILE_SHARE_READ',
                              'name': 'data.bin'},
                  'type': 'stream'},
 'res_stream_2': {'payload': {'buffer': 'POSIX-originated payload'},
                  'type': 'stream'}}


In [3]:
import ctypes
import subprocess
import os

# Подключаем стандартную libc для честного выделения виртуальных страниц
libc = ctypes.CDLL(None)
libc.mmap.restype = ctypes.c_void_p
libc.mmap.argtypes = [ctypes.c_void_p, ctypes.c_size_t, ctypes.c_int, ctypes.c_int, ctypes.c_int, ctypes.c_int64]

PROT_READ = 0x1
PROT_WRITE = 0x2
MAP_PRIVATE = 0x02
MAP_ANONYMOUS = 0x20

# =====================================================================
# 1. МЕТА-ЯДРО (LISP-БАreader/evaluator над реальными страницами памяти)
# =====================================================================
class UniversalMemoryKernel:
    def __init__(self):
        self.allocations = {} # raw_addr -> {'size': int, 'pages': int}

    def eval(self, expr):
        if not isinstance(expr, list):
            return expr

        op = expr[0]
        args = expr[1:]

        # Примитив: Выделение реальной анонимной памяти (Zero-copy базис)
        if op == ":vm/alloc":
            size, prot_flags = args
            # Реальный POSIX mmap: возвращает честный 64-битный адрес памяти в адресном пространстве процесса
            addr = libc.mmap(0, size, PROT_READ | PROT_WRITE, MAP_PRIVATE | MAP_ANONYMOUS, -1, 0)
            if addr == -1 or not addr:
                return 0

            self.allocations[addr] = {"size": size}
            return addr

        # Примитив: Запись байт в память
        elif op == ":vm/write_bytes":
            addr, offset, data_bytes = args
            if addr not in self.allocations:
                # Разрешаем прямую запись по валидному указателю
                pass

            target_ptr = addr + offset
            ctypes.memmove(target_ptr, data_bytes, len(data_bytes))
            return len(data_bytes)

        # Примитив: Чтение системных метрик / шины
        elif op == ":sys/query":
            metric_id = args[0]
            metrics = {0: 1920, 1: 1080} # SM_CXSCREEN, SM_CYSCREEN
            return metrics.get(metric_id, -1)

# Инициализируем рантайм
KERNEL = UniversalMemoryKernel()

# =====================================================================
# 2. МЕТА-МАКРОСЫ (Трансляция вызовов Win32 C-ABI -> Lisp S-Expressions)
# =====================================================================
HOOK_VIRTUAL_ALLOC = ctypes.CFUNCTYPE(ctypes.c_void_p, ctypes.c_void_p, ctypes.c_size_t, ctypes.c_uint32, ctypes.c_uint32)
HOOK_WRITE_PROCESS_MEM = ctypes.CFUNCTYPE(ctypes.c_int, ctypes.c_void_p, ctypes.c_void_p, ctypes.c_char_p, ctypes.c_size_t, ctypes.POINTER(ctypes.c_size_t))
HOOK_GET_SYSTEM_METRICS = ctypes.CFUNCTYPE(ctypes.c_int, ctypes.c_int)

@HOOK_VIRTUAL_ALLOC
def win32_VirtualAlloc(lpAddress, dwSize, flAllocationType, flProtect):
    ast = [":vm/alloc", dwSize, flProtect]
    raw_addr = KERNEL.eval(ast)
    return raw_addr # Возвращаем integer адрес для c_void_p

@HOOK_WRITE_PROCESS_MEM
def win32_WriteProcessMemory(hProcess, lpBaseAddress, lpBuffer, nSize, lpNumberOfBytesWritten):
    raw_bytes = ctypes.string_at(lpBuffer, nSize)
    ast = [":vm/write_bytes", lpBaseAddress, 0, raw_bytes]
    bytes_written = KERNEL.eval(ast)
    if lpNumberOfBytesWritten:
        lpNumberOfBytesWritten.contents.value = bytes_written
    return 1 if bytes_written > 0 else 0

@HOOK_GET_SYSTEM_METRICS
def win32_GetSystemMetrics(nIndex):
    ast = [":sys/query", nIndex]
    return KERNEL.eval(ast)

# =====================================================================
# 3. НАСТОЯЩИЙ C-БИНАРНИК (Имитация логики Windows-программы)
# =====================================================================
c_code = """
#include <stdint.h>
#include <stddef.h>
#include <string.h>

typedef struct {
    void* (*VirtualAlloc)(void*, size_t, uint32_t, uint32_t);
    int   (*WriteProcessMemory)(void*, void*, const char*, size_t, size_t*);
    int   (*GetSystemMetrics)(int);
} Win32ImportTable;

int run_windows_payload(Win32ImportTable* win32) {
    // 1. Запрос метрик экрана (Win32 API)
    int width = win32->GetSystemMetrics(0);
    int height = win32->GetSystemMetrics(1);

    // 2. Выделение памяти через Win32 VirtualAlloc
    void* pMem = win32->VirtualAlloc(NULL, 128, 0x1000, 0x04);
    if (!pMem) return -1;

    // 3. Запись payload через WriteProcessMemory
    const char msg[] = "Real Native Execution via Lisp Meta-Runtime!";
    size_t written = 0;
    win32->WriteProcessMemory((void*)0xFFFFFFFF, pMem, msg, sizeof(msg), &written);

    return (width == 1920 && height == 1080 && written == sizeof(msg)) ? 0 : 1;
}
"""

with open("payload.c", "w") as f:
    f.write(c_code)

# Компилируем C-код в shared library
subprocess.run(["gcc", "-shared", "-fPIC", "-O2", "payload.c", "-o", "libpayload.so"], check=True)

# Загружаем библиотеку
lib = ctypes.CDLL("./libpayload.so")

class Win32ImportTable(ctypes.Structure):
    _fields_ = [
        ("VirtualAlloc", HOOK_VIRTUAL_ALLOC),
        ("WriteProcessMemory", HOOK_WRITE_PROCESS_MEM),
        ("GetSystemMetrics", HOOK_GET_SYSTEM_METRICS)
    ]

# Собираем таблицу импорта
import_table = Win32ImportTable(
    VirtualAlloc=win32_VirtualAlloc,
    WriteProcessMemory=win32_WriteProcessMemory,
    GetSystemMetrics=win32_GetSystemMetrics
)

# Запускаем бинарник
lib.run_windows_payload.argtypes = [ctypes.POINTER(Win32ImportTable)]
lib.run_windows_payload.restype = ctypes.c_int

status = lib.run_windows_payload(ctypes.byref(import_table))

print(f"[*] C Payload Execution Status: {'SUCCESS (0)' if status == 0 else 'FAILED'}")
print("\n--- Проверяем содержимое реальной памяти mmap через ядро ---")
for addr, data in KERNEL.allocations.items():
    raw_content = ctypes.string_at(addr, data["size"])
    print(f"Allocated Address: 0x{addr:X} (Size: {data['size']} bytes)")
    print(f"Содержимое памяти: {raw_content[:45].decode('ascii')}")

[*] C Payload Execution Status: SUCCESS (0)

--- Проверяем содержимое реальной памяти mmap через ядро ---
Allocated Address: 0x7AFA53CA2000 (Size: 128 bytes)
Содержимое памяти: Real Native Execution via Lisp Meta-Runtime! 


In [1]:
import subprocess

# 1. Компилируем Windows PE-бинарник
win_c_code = """
#include <windows.h>

void mainCRTStartup() {
    HANDLE hOut = GetStdHandle(STD_OUTPUT_HANDLE);
    const char msg[] = "[WIN_EXE] Real Windows PE binary executed inside Linux without Wine!\\n";
    DWORD written = 0;
    WriteConsoleA(hOut, msg, (DWORD)(sizeof(msg) - 1), &written, NULL);
    ExitProcess(42);
}
"""

with open("win_app.c", "w") as f:
    f.write(win_c_code)

subprocess.run(["apt-get", "install", "-y", "-qq", "gcc-mingw-w64-x86-64"], check=True)
subprocess.run([
    "x86_64-w64-mingw32-gcc",
    "-nostdlib", "-fno-ident", "-O2",
    "win_app.c", "-o", "win_app.exe",
    "-lkernel32"
], check=True)

# 2. Пишем нативный C-лоадер (с setjmp/longjmp для корректного перехвата ExitProcess)
native_loader_c = """
#define _GNU_SOURCE
#include <stdio.h>
#include <stdlib.h>
#include <stdint.h>
#include <string.h>
#include <sys/mman.h>
#include <unistd.h>
#include <setjmp.h>

static jmp_buf g_exit_env;
static int g_captured_exit_code = -1;

// --- Хуки с Microsoft x64 ABI calling convention ---
__attribute__((ms_abi))
void* hook_GetStdHandle(uint32_t nStdHandle) {
    return (void*)0x100;
}

__attribute__((ms_abi))
int hook_WriteConsoleA(void* hConsole, const char* lpBuffer, uint32_t nChars, uint32_t* lpWritten, void* lpReserved) {
    write(STDOUT_FILENO, "\\n>>> [INTERCEPTED WriteConsoleA]: ", 35);
    write(STDOUT_FILENO, lpBuffer, nChars);
    if (lpWritten) *lpWritten = nChars;
    return 1;
}

__attribute__((ms_abi))
void hook_ExitProcess(uint32_t uExitCode) {
    g_captured_exit_code = (int)uExitCode;
    longjmp(g_exit_env, 1); // Мгновенный возврат в лоадер без падения в мусорную память!
}

int main(int argc, char** argv) {
    FILE* f = fopen("win_app.exe", "rb");
    if (!f) return 1;
    fseek(f, 0, SEEK_END);
    size_t size = ftell(f);
    fseek(f, 0, SEEK_SET);
    uint8_t* pe_data = malloc(size);
    fread(pe_data, 1, size, f);
    fclose(f);

    uint32_t e_lfanew = *(uint32_t*)(pe_data + 0x3C);
    uint16_t num_sections = *(uint16_t*)(pe_data + e_lfanew + 0x6);
    uint8_t* opt_header = pe_data + e_lfanew + 0x18;

    uint64_t pref_base = *(uint64_t*)(opt_header + 0x18);
    uint32_t entry_rva = *(uint32_t*)(opt_header + 0x10);
    uint32_t image_size = *(uint32_t*)(opt_header + 0x38);
    uint32_t headers_size = *(uint32_t*)(opt_header + 0x3C);
    uint8_t* data_dirs = opt_header + 0x70;

    uint32_t import_rva = *(uint32_t*)(data_dirs + 0x8);
    uint32_t reloc_rva = *(uint32_t*)(data_dirs + 0x28);
    uint32_t reloc_size = *(uint32_t*)(data_dirs + 0x2C);

    uint8_t* image_base = (uint8_t*)mmap(NULL, image_size, PROT_READ | PROT_WRITE | PROT_EXEC, MAP_PRIVATE | MAP_ANONYMOUS, -1, 0);
    printf("[+] Allocated Memory Base: %p (Preferable: %p)\\n", image_base, (void*)pref_base);

    memcpy(image_base, pe_data, headers_size);

    uint8_t* sec_header = opt_header + *(uint16_t*)(pe_data + e_lfanew + 0x14);
    for (int i = 0; i < num_sections; i++) {
        uint8_t* sec = sec_header + i * 40;
        uint32_t v_addr = *(uint32_t*)(sec + 12);
        uint32_t raw_size = *(uint32_t*)(sec + 16);
        uint32_t raw_ptr = *(uint32_t*)(sec + 20);
        if (raw_size > 0) {
            memcpy(image_base + v_addr, pe_data + raw_ptr, raw_size);
        }
    }

    // Релокации
    int64_t delta = (int64_t)image_base - (int64_t)pref_base;
    if (reloc_rva && delta != 0) {
        uint32_t offset = 0;
        while (offset < reloc_size) {
            uint32_t page_rva = *(uint32_t*)(pe_data + reloc_rva + offset);
            uint32_t block_size = *(uint32_t*)(pe_data + reloc_rva + offset + 4);
            if (!block_size) break;
            int count = (block_size - 8) / 2;
            uint16_t* entries = (uint16_t*)(pe_data + reloc_rva + offset + 8);
            for (int j = 0; j < count; j++) {
                if ((entries[j] >> 12) == 10) { // DIR64
                    uint64_t* target = (uint64_t*)(image_base + page_rva + (entries[j] & 0xFFF));
                    *target += delta;
                }
            }
            offset += block_size;
        }
        printf("[+] Applied base relocations successfully.\\n");
    }

    // Патчинг IAT
    if (import_rva) {
        uint8_t* desc = image_base + import_rva;
        while (1) {
            uint32_t orig_thunk = *(uint32_t*)desc;
            uint32_t first_thunk = *(uint32_t*)(desc + 16);
            if (!orig_thunk && !first_thunk) break;
            uint32_t thunk_rva = orig_thunk ? orig_thunk : first_thunk;
            uint64_t* thunk = (uint64_t*)(image_base + thunk_rva);
            uint64_t* iat = (uint64_t*)(image_base + first_thunk);

            while (*thunk) {
                if (!(*thunk & (1ULL << 63))) {
                    char* name = (char*)(image_base + *thunk + 2);
                    if (strcmp(name, "GetStdHandle") == 0) *iat = (uint64_t)hook_GetStdHandle;
                    else if (strcmp(name, "WriteConsoleA") == 0) *iat = (uint64_t)hook_WriteConsoleA;
                    else if (strcmp(name, "ExitProcess") == 0) *iat = (uint64_t)hook_ExitProcess;
                    printf("[+] Patched IAT: %s -> Hook: %p\\n", name, (void*)*iat);
                }
                thunk++;
                iat++;
            }
            desc += 20;
        }
    }

    // Точка входа с ms_abi конвенцией
    typedef __attribute__((ms_abi)) void (*WinEntry_t)();
    WinEntry_t entry = (WinEntry_t)(image_base + entry_rva);

    printf("[*] Executing Windows EntryPoint at %p...\\n", entry);

    if (setjmp(g_exit_env) == 0) {
        entry();
    }

    printf("\\n[*] Process terminated cleanly! Captured ExitCode: %d\\n", g_captured_exit_code);
    return 0;
}
"""

with open("loader.c", "w") as f:
    f.write(native_loader_c)

# Компилируем нативный лоадер хоста
subprocess.run(["gcc", "-O2", "loader.c", "-o", "native_loader"], check=True)

# Запускаем лоадер
res = subprocess.run(["./native_loader"], capture_output=True, text=True)
print(res.stdout)
if res.stderr:
    print("STDERR:", res.stderr)


>>> [INTERCEPTED WriteConsoleA]:  [WIN_EXE] Real Windows PE binary executed inside Linux without Wine!
[+] Allocated Memory Base: 0x7befdeab6000 (Preferable: 0x140000000)
[+] Patched IAT: ExitProcess -> Hook: 0x59d0172cf760
[+] Patched IAT: GetStdHandle -> Hook: 0x59d0172cf670
[+] Patched IAT: WriteConsoleA -> Hook: 0x59d0172cf680
[*] Executing Windows EntryPoint at 0x7befdeab7000...

[*] Process terminated cleanly! Captured ExitCode: 42



In [2]:
import subprocess
import os

# 1. Скачиваем официальный релизный бинарник Windows x64 (7-Zip Standalone Console x64)
# Это 100% реальный, сложный Windows PE x64 бинарник, скомпилированный MSVC.
subprocess.run(["wget", "-q", "-O", "7z_win.zip", "https://www.7-zip.org/a/7z2408-extra.7z"], check=False)
if not os.path.exists("7z_win.zip"):
    # Фолбек: соберем реальный MSVC-подобный бинарник со сложным CRT и C++ кодом
    pass

# Для 100% надежности теста в Colab скачаем официальный UPX для Windows x64 или 7za.exe
subprocess.run(["apt-get", "install", "-y", "-qq", "p7zip-full"], check=True)
subprocess.run(["wget", "-q", "https://www.7-zip.org/a/7z2408-extra.7z"], check=True)
subprocess.run(["7z", "x", "-y", "7z2408-extra.7z", "x64/7za.exe"], check=True)
subprocess.run(["cp", "x64/7za.exe", "target_app.exe"], check=True)

print("[+] Target Windows binary 'target_app.exe' (7-Zip Standalone x64) prepared.")

# 2. Пишем полноценный Windows-to-Linux Meta-Loader с TEB/PEB и CRT-базисом
loader_source = """
#define _GNU_SOURCE
#include <stdio.h>
#include <stdlib.h>
#include <stdint.h>
#include <string.h>
#include <sys/mman.h>
#include <unistd.h>
#include <setjmp.h>
#include <sys/syscall.h>
#include <asm/prctl.h>
#include <wchar.h>

static jmp_buf g_exit_env;
static int g_exit_code = 0;
static uint8_t* g_image_base = NULL;

// ====================================================================
// 1. СТРУКТУРЫ TEB / PEB ДЛЯ GS-СЕГМЕНТА WINDOWS x64
// ====================================================================
typedef struct {
    uint8_t Reserved1[2];
    uint8_t BeingDebugged;
    uint8_t Reserved2[21];
    void* ProcessHeap;
    void* FastPebLock;
    void* AtlThunkSListPtr;
    void* IFEOKey;
    void* CrossProcessFlags;
    void* UserSharedInfoPtr;
    void* SystemReserved;
    void* AtlThunkSListPtr32;
    void* ApiSetMap;
    uint32_t TlsExpansionCounter;
    void* TlsBitmap;
    uint32_t TlsBitmapBits[2];
    void* ReadOnlySharedMemoryBase;
    void* SharedData;
    void** ReadOnlyStaticServerData;
    void* AnsiCodePageData;
    void* OemCodePageData;
    void* UnicodeCaseTableData;
    uint32_t NumberOfProcessors;
    uint32_t NtGlobalFlag;
} __attribute__((packed)) WIN_PEB;

typedef struct {
    void* ExceptionList;
    void* StackBase;
    void* StackLimit;
    void* SubSystemTib;
    void* FiberData;
    void* ArbitraryUserPointer;
    void* Self;                  // 0x30: Адрес самого TEB
    void* EnvironmentPointer;
    uint64_t ProcessId;
    uint64_t ThreadId;
    void* RpcHandle;
    void* Tls;
    WIN_PEB* ProcessEnvironmentBlock; // 0x60: Указатель на PEB
} __attribute__((packed)) WIN_TEB;

static WIN_TEB g_teb;
static WIN_PEB g_peb;

// ====================================================================
// 2. РЕАЛИЗАЦИЯ БАЗОВЫХ ВЫЗОВОВ KERNEL32 / NTDLL
// ====================================================================

#define MS_ABI __attribute__((ms_abi))

MS_ABI void* hook_GetModuleHandleW(const wchar_t* lpModuleName) {
    return g_image_base; // NULL или имя модуля -> возвращаем ImageBase
}

MS_ABI void* hook_GetModuleHandleA(const char* lpModuleName) {
    return g_image_base;
}

MS_ABI void* hook_GetCommandLineW(void) {
    static wchar_t cmd[] = L"7za.exe --help";
    return cmd;
}

MS_ABI void* hook_GetCommandLineA(void) {
    static char cmd[] = "7za.exe --help";
    return cmd;
}

MS_ABI void* hook_GetStdHandle(uint32_t nStdHandle) {
    return (void*)(uint64_t)(0x100 + (nStdHandle & 0xFF));
}

MS_ABI void* hook_GetProcessHeap(void) {
    return (void*)0x50000;
}

MS_ABI void* hook_HeapAlloc(void* hHeap, uint32_t dwFlags, size_t dwBytes) {
    return malloc(dwBytes);
}

MS_ABI int hook_HeapFree(void* hHeap, uint32_t dwFlags, void* lpMem) {
    free(lpMem);
    return 1;
}

MS_ABI void* hook_HeapReAlloc(void* hHeap, uint32_t dwFlags, void* lpMem, size_t dwBytes) {
    return realloc(lpMem, dwBytes);
}

MS_ABI int hook_WriteFile(void* hFile, const void* lpBuffer, uint32_t nNumberOfBytesToWrite, uint32_t* lpNumberOfBytesWritten, void* lpOverlapped) {
    ssize_t res = write(STDOUT_FILENO, lpBuffer, nNumberOfBytesToWrite);
    if (lpNumberOfBytesWritten && res > 0) *lpNumberOfBytesWritten = (uint32_t)res;
    return 1;
}

MS_ABI int hook_WriteConsoleW(void* hConsole, const wchar_t* lpBuffer, uint32_t nChars, uint32_t* lpWritten, void* lpReserved) {
    char buf[4096];
    size_t len = wcstombs(buf, lpBuffer, sizeof(buf) - 1);
    if (len != (size_t)-1) {
        write(STDOUT_FILENO, buf, len);
    }
    if (lpWritten) *lpWritten = nChars;
    return 1;
}

MS_ABI int hook_WriteConsoleA(void* hConsole, const char* lpBuffer, uint32_t nChars, uint32_t* lpWritten, void* lpReserved) {
    write(STDOUT_FILENO, lpBuffer, nChars);
    if (lpWritten) *lpWritten = nChars;
    return 1;
}

MS_ABI void hook_ExitProcess(uint32_t uExitCode) {
    g_exit_code = (int)uExitCode;
    longjmp(g_exit_env, 1);
}

MS_ABI void* hook_SetUnhandledExceptionFilter(void* lpFilter) {
    return NULL;
}

MS_ABI uint32_t hook_GetLastError(void) {
    return 0;
}

MS_ABI void hook_SetLastError(uint32_t dwErr) {}

MS_ABI int hook_GetConsoleMode(void* hConsole, uint32_t* lpMode) {
    if (lpMode) *lpMode = 0x0007;
    return 1;
}

MS_ABI int hook_SetConsoleMode(void* hConsole, uint32_t dwMode) {
    return 1;
}

MS_ABI uint32_t hook_GetFileType(void* hFile) {
    return 0x0002; // FILE_TYPE_CHAR (Консоль)
}

MS_ABI uint32_t hook_GetConsoleCP(void) { return 65001; } // UTF-8
MS_ABI uint32_t hook_GetConsoleOutputCP(void) { return 65001; }

MS_ABI int hook_QueryPerformanceCounter(uint64_t* lpPerformanceCount) {
    if (lpPerformanceCount) *lpPerformanceCount = 1000000;
    return 1;
}

MS_ABI int hook_QueryPerformanceFrequency(uint64_t* lpFrequency) {
    if (lpFrequency) *lpFrequency = 10000000;
    return 1;
}

MS_ABI uint32_t hook_GetCurrentProcessId(void) { return getpid(); }
MS_ABI uint32_t hook_GetCurrentThreadId(void) { return getpid(); }

MS_ABI void hook_GetSystemTimeAsFileTime(void* lpSystemTimeAsFileTime) {
    memset(lpSystemTimeAsFileTime, 0, 8);
}

MS_ABI void hook_InitializeCriticalSection(void* lpCrit) {}
MS_ABI void hook_EnterCriticalSection(void* lpCrit) {}
MS_ABI void hook_LeaveCriticalSection(void* lpCrit) {}
MS_ABI void hook_DeleteCriticalSection(void* lpCrit) {}

MS_ABI void* hook_VirtualAlloc(void* lpAddr, size_t size, uint32_t type, uint32_t protect) {
    return mmap(NULL, size, PROT_READ | PROT_WRITE, MAP_PRIVATE | MAP_ANONYMOUS, -1, 0);
}

MS_ABI int hook_VirtualFree(void* lpAddr, size_t size, uint32_t type) {
    munmap(lpAddr, size);
    return 1;
}

MS_ABI void* hook_TlsAlloc(void) { return (void*)1; }
MS_ABI void* hook_TlsGetValue(uint32_t idx) { return NULL; }
MS_ABI int hook_TlsSetValue(uint32_t idx, void* val) { return 1; }
MS_ABI int hook_TlsFree(uint32_t idx) { return 1; }

// Таблица символов
typedef struct {
    const char* name;
    void* func;
} SymbolEntry;

static SymbolEntry g_symbols[] = {
    {"GetModuleHandleW", hook_GetModuleHandleW},
    {"GetModuleHandleA", hook_GetModuleHandleA},
    {"GetCommandLineW", hook_GetCommandLineW},
    {"GetCommandLineA", hook_GetCommandLineA},
    {"GetStdHandle", hook_GetStdHandle},
    {"GetProcessHeap", hook_GetProcessHeap},
    {"HeapAlloc", hook_HeapAlloc},
    {"HeapFree", hook_HeapFree},
    {"HeapReAlloc", hook_HeapReAlloc},
    {"WriteFile", hook_WriteFile},
    {"WriteConsoleW", hook_WriteConsoleW},
    {"WriteConsoleA", hook_WriteConsoleA},
    {"ExitProcess", hook_ExitProcess},
    {"SetUnhandledExceptionFilter", hook_SetUnhandledExceptionFilter},
    {"GetLastError", hook_GetLastError},
    {"SetLastError", hook_SetLastError},
    {"GetConsoleMode", hook_GetConsoleMode},
    {"SetConsoleMode", hook_SetConsoleMode},
    {"GetFileType", hook_GetFileType},
    {"GetConsoleCP", hook_GetConsoleCP},
    {"GetConsoleOutputCP", hook_GetConsoleOutputCP},
    {"QueryPerformanceCounter", hook_QueryPerformanceCounter},
    {"QueryPerformanceFrequency", hook_QueryPerformanceFrequency},
    {"GetCurrentProcessId", hook_GetCurrentProcessId},
    {"GetCurrentThreadId", hook_GetCurrentThreadId},
    {"GetSystemTimeAsFileTime", hook_GetSystemTimeAsFileTime},
    {"InitializeCriticalSection", hook_InitializeCriticalSection},
    {"EnterCriticalSection", hook_EnterCriticalSection},
    {"LeaveCriticalSection", hook_LeaveCriticalSection},
    {"DeleteCriticalSection", hook_DeleteCriticalSection},
    {"VirtualAlloc", hook_VirtualAlloc},
    {"VirtualFree", hook_VirtualFree},
    {"TlsAlloc", hook_TlsAlloc},
    {"TlsGetValue", hook_TlsGetValue},
    {"TlsSetValue", hook_TlsSetValue},
    {"TlsFree", hook_TlsFree},
    {NULL, NULL}
};

void* resolve_symbol(const char* name) {
    for (int i = 0; g_symbols[i].name != NULL; i++) {
        if (strcmp(g_symbols[i].name, name) == 0) return g_symbols[i].func;
    }
    return NULL;
}

// Заглушка для неизвестных функций
MS_ABI uint64_t generic_stub(void) {
    return 0;
}

// ====================================================================
// 3. ЗАГРУЗЧИК PE И ЗАПУСК
// ====================================================================
int main(int argc, char** argv) {
    FILE* f = fopen("target_app.exe", "rb");
    if (!f) {
        printf("[-] Failed to open target_app.exe\\n");
        return 1;
    }
    fseek(f, 0, SEEK_END);
    size_t size = ftell(f);
    fseek(f, 0, SEEK_SET);
    uint8_t* pe_data = malloc(size);
    fread(pe_data, 1, size, f);
    fclose(f);

    uint32_t e_lfanew = *(uint32_t*)(pe_data + 0x3C);
    uint16_t num_sections = *(uint16_t*)(pe_data + e_lfanew + 0x6);
    uint8_t* opt_header = pe_data + e_lfanew + 0x18;

    uint64_t pref_base = *(uint64_t*)(opt_header + 0x18);
    uint32_t entry_rva = *(uint32_t*)(opt_header + 0x10);
    uint32_t image_size = *(uint32_t*)(opt_header + 0x38);
    uint32_t headers_size = *(uint32_t*)(opt_header + 0x3C);
    uint8_t* data_dirs = opt_header + 0x70;

    uint32_t import_rva = *(uint32_t*)(data_dirs + 0x8);
    uint32_t reloc_rva = *(uint32_t*)(data_dirs + 0x28);
    uint32_t reloc_size = *(uint32_t*)(data_dirs + 0x2C);

    g_image_base = (uint8_t*)mmap((void*)pref_base, image_size, PROT_READ | PROT_WRITE | PROT_EXEC, MAP_PRIVATE | MAP_ANONYMOUS, -1, 0);
    if (g_image_base == MAP_FAILED) {
        g_image_base = (uint8_t*)mmap(NULL, image_size, PROT_READ | PROT_WRITE | PROT_EXEC, MAP_PRIVATE | MAP_ANONYMOUS, -1, 0);
    }
    printf("[+] Allocated Image Base: %p (Original: %p)\\n", g_image_base, (void*)pref_base);

    memcpy(g_image_base, pe_data, headers_size);

    uint8_t* sec_header = opt_header + *(uint16_t*)(pe_data + e_lfanew + 0x14);
    for (int i = 0; i < num_sections; i++) {
        uint8_t* sec = sec_header + i * 40;
        uint32_t v_addr = *(uint32_t*)(sec + 12);
        uint32_t raw_size = *(uint32_t*)(sec + 16);
        uint32_t raw_ptr = *(uint32_t*)(sec + 20);
        if (raw_size > 0) {
            memcpy(g_image_base + v_addr, pe_data + raw_ptr, raw_size);
        }
    }

    // Релокации
    int64_t delta = (int64_t)g_image_base - (int64_t)pref_base;
    if (reloc_rva && delta != 0) {
        uint32_t offset = 0;
        while (offset < reloc_size) {
            uint32_t page_rva = *(uint32_t*)(pe_data + reloc_rva + offset);
            uint32_t block_size = *(uint32_t*)(pe_data + reloc_rva + offset + 4);
            if (!block_size) break;
            int count = (block_size - 8) / 2;
            uint16_t* entries = (uint16_t*)(pe_data + reloc_rva + offset + 8);
            for (int j = 0; j < count; j++) {
                if ((entries[j] >> 12) == 10) {
                    uint64_t* target = (uint64_t*)(g_image_base + page_rva + (entries[j] & 0xFFF));
                    *target += delta;
                }
            }
            offset += block_size;
        }
    }

    // Разрешение IAT
    if (import_rva) {
        uint8_t* desc = g_image_base + import_rva;
        while (1) {
            uint32_t orig_thunk = *(uint32_t*)desc;
            uint32_t first_thunk = *(uint32_t*)(desc + 16);
            if (!orig_thunk && !first_thunk) break;
            uint32_t thunk_rva = orig_thunk ? orig_thunk : first_thunk;
            uint64_t* thunk = (uint64_t*)(g_image_base + thunk_rva);
            uint64_t* iat = (uint64_t*)(g_image_base + first_thunk);

            while (*thunk) {
                if (!(*thunk & (1ULL << 63))) {
                    char* name = (char*)(g_image_base + *thunk + 2);
                    void* resolved = resolve_symbol(name);
                    if (resolved) {
                        *iat = (uint64_t)resolved;
                    } else {
                        *iat = (uint64_t)generic_stub;
                    }
                }
                thunk++;
                iat++;
            }
            desc += 20;
        }
    }

    // Настраиваем TEB/PEB и GS-регистр процессора
    memset(&g_teb, 0, sizeof(g_teb));
    memset(&g_peb, 0, sizeof(g_peb));
    g_teb.Self = &g_teb;
    g_teb.ProcessEnvironmentBlock = &g_peb;
    g_peb.ProcessHeap = (void*)0x50000;

    // Устанавливаем базу сегмента GS на наш TEB через сисколл Linux
    syscall(SYS_arch_prctl, ARCH_SET_GS, &g_teb);
    printf("[+] Thread Environment Block (TEB) attached to GS register.\\n");

    // Запуск
    typedef MS_ABI void (*WinEntry_t)(void);
    WinEntry_t entry = (WinEntry_t)(g_image_base + entry_rva);

    printf("[*] Launching native Windows PE at %p...\\n\\n", entry);

    if (setjmp(g_exit_env) == 0) {
        entry();
    }

    printf("\\n[*] Windows Application cleanly terminated! Exit code: %d\\n", g_exit_code);
    return 0;
}
"""

with open("win_runner.c", "w") as f:
    f.write(loader_source)

subprocess.run(["gcc", "-O2", "win_runner.c", "-o", "win_runner"], check=True)
res = subprocess.run(["./win_runner"], capture_output=True, text=True)
print(res.stdout)
if res.stderr:
    print("STDERR:", res.stderr)

CalledProcessError: Command '['wget', '-q', 'https://www.7-zip.org/a/7z2408-extra.7z']' returned non-zero exit status 8.

In [3]:
import subprocess
import os

# 1. Устанавливаем MinGW C++
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "g++-mingw-w64-x86-64"], check=True)

# 2. Пишем сложное Windows C++ приложение (настоящий CRT, ООП, dynamic heap)
cpp_code = """
#include <windows.h>
#include <string>
#include <vector>
#include <sstream>

class WindowsEngine {
private:
    std::string title;
    std::vector<std::string> modules;

public:
    WindowsEngine(const std::string& name) : title(name) {
        modules.push_back("MemoryManager");
        modules.push_back("IO_Stream");
        modules.push_back("DynamicAST_Hook");
    }

    void run() {
        HANDLE hOut = GetStdHandle(STD_OUTPUT_HANDLE);
        std::stringstream ss;
        ss << "\\n========================================\\n";
        ss << "  [WIN_APP] " << title << "\\n";
        ss << "  Active Modules: " << modules.size() << "\\n";
        for (size_t i = 0; i < modules.size(); ++i) {
            ss << "   - Module [" << i << "]: " << modules[i] << "\\n";
        }
        ss << "========================================\\n";

        std::string out = ss.str();
        DWORD written = 0;
        WriteFile(hOut, out.c_str(), (DWORD)out.size(), &written, NULL);
    }
};

int main() {
    WindowsEngine* engine = new WindowsEngine("Native Windows C++ Runtime via Meta-Loader");
    engine->run();
    delete engine;
    ExitProcess(0);
}
"""

with open("target_app.cpp", "w") as f:
    f.write(cpp_code)

# Компилируем реальный Windows PE x64 C++ бинарник со стандартным CRT
subprocess.run([
    "x86_64-w64-mingw32-g++",
    "-O2",
    "-static-libgcc",
    "-static-libstdc++",
    "target_app.cpp",
    "-o", "target_app.exe",
    "-lkernel32"
], check=True)

print("[+] Target Windows C++ PE binary 'target_app.exe' successfully compiled.")

# ====================================================================
# 3. НАШ НАРОДНЫЙ LINUX PE-ЛОАДЕР С TEB/PEB И KERNEL32 РЕЗОЛВЕРОМ
# ====================================================================
loader_source = """
#define _GNU_SOURCE
#include <stdio.h>
#include <stdlib.h>
#include <stdint.h>
#include <string.h>
#include <sys/mman.h>
#include <unistd.h>
#include <setjmp.h>
#include <sys/syscall.h>
#include <asm/prctl.h>

static jmp_buf g_exit_env;
static int g_exit_code = 0;
static uint8_t* g_image_base = NULL;

// TEB / PEB для Windows x64 (GS-регистр)
typedef struct {
    uint8_t Reserved[24];
    void* ProcessHeap;
} WIN_PEB;

typedef struct {
    void* ExceptionList;
    void* StackBase;
    void* StackLimit;
    void* SubSystemTib;
    void* FiberData;
    void* ArbitraryUserPointer;
    void* Self;                  // 0x30: Адрес самого TEB
    void* EnvironmentPointer;
    uint64_t ProcessId;
    uint64_t ThreadId;
    void* RpcHandle;
    void* Tls;
    WIN_PEB* ProcessEnvironmentBlock; // 0x60: Адрес PEB
} __attribute__((packed)) WIN_TEB;

static WIN_TEB g_teb;
static WIN_PEB g_peb;

#define MS_ABI __attribute__((ms_abi))

MS_ABI void* hook_GetModuleHandleW(const void* name) { return g_image_base; }
MS_ABI void* hook_GetModuleHandleA(const char* name) { return g_image_base; }
MS_ABI void* hook_GetStdHandle(uint32_t nStdHandle) { return (void*)0x100; }
MS_ABI void* hook_GetProcessHeap(void) { return (void*)0x50000; }

MS_ABI void* hook_HeapAlloc(void* hHeap, uint32_t dwFlags, size_t dwBytes) {
    return malloc(dwBytes);
}

MS_ABI int hook_HeapFree(void* hHeap, uint32_t dwFlags, void* lpMem) {
    free(lpMem);
    return 1;
}

MS_ABI void* hook_HeapReAlloc(void* hHeap, uint32_t dwFlags, void* lpMem, size_t dwBytes) {
    return realloc(lpMem, dwBytes);
}

MS_ABI int hook_WriteFile(void* hFile, const void* lpBuffer, uint32_t nNumberOfBytesToWrite, uint32_t* lpNumberOfBytesWritten, void* lpOverlapped) {
    ssize_t res = write(STDOUT_FILENO, lpBuffer, nNumberOfBytesToWrite);
    if (lpNumberOfBytesWritten && res > 0) *lpNumberOfBytesWritten = (uint32_t)res;
    return 1;
}

MS_ABI void hook_ExitProcess(uint32_t uExitCode) {
    g_exit_code = (int)uExitCode;
    longjmp(g_exit_env, 1);
}

MS_ABI void* hook_SetUnhandledExceptionFilter(void* lpFilter) { return NULL; }
MS_ABI uint32_t hook_GetLastError(void) { return 0; }
MS_ABI void hook_SetLastError(uint32_t dwErr) {}
MS_ABI uint32_t hook_GetCurrentProcessId(void) { return getpid(); }
MS_ABI uint32_t hook_GetCurrentThreadId(void) { return getpid(); }
MS_ABI void hook_GetSystemTimeAsFileTime(void* lpTime) { memset(lpTime, 0, 8); }
MS_ABI void hook_InitializeCriticalSection(void* lpCrit) {}
MS_ABI void hook_EnterCriticalSection(void* lpCrit) {}
MS_ABI void hook_LeaveCriticalSection(void* lpCrit) {}
MS_ABI void hook_DeleteCriticalSection(void* lpCrit) {}

MS_ABI void* hook_VirtualAlloc(void* lpAddr, size_t size, uint32_t type, uint32_t protect) {
    return mmap(NULL, size, PROT_READ | PROT_WRITE, MAP_PRIVATE | MAP_ANONYMOUS, -1, 0);
}

MS_ABI int hook_VirtualFree(void* lpAddr, size_t size, uint32_t type) {
    munmap(lpAddr, size);
    return 1;
}

MS_ABI void* hook_TlsAlloc(void) { return (void*)1; }
MS_ABI void* hook_TlsGetValue(uint32_t idx) { return NULL; }
MS_ABI int hook_TlsSetValue(uint32_t idx, void* val) { return 1; }
MS_ABI int hook_TlsFree(uint32_t idx) { return 1; }
MS_ABI void* hook_GetProcAddress(void* hMod, const char* name) { return NULL; }

// Таблица символов
typedef struct {
    const char* name;
    void* func;
} SymbolEntry;

static SymbolEntry g_symbols[] = {
    {"GetModuleHandleW", hook_GetModuleHandleW},
    {"GetModuleHandleA", hook_GetModuleHandleA},
    {"GetStdHandle", hook_GetStdHandle},
    {"GetProcessHeap", hook_GetProcessHeap},
    {"HeapAlloc", hook_HeapAlloc},
    {"HeapFree", hook_HeapFree},
    {"HeapReAlloc", hook_HeapReAlloc},
    {"WriteFile", hook_WriteFile},
    {"ExitProcess", hook_ExitProcess},
    {"SetUnhandledExceptionFilter", hook_SetUnhandledExceptionFilter},
    {"GetLastError", hook_GetLastError},
    {"SetLastError", hook_SetLastError},
    {"GetCurrentProcessId", hook_GetCurrentProcessId},
    {"GetCurrentThreadId", hook_GetCurrentThreadId},
    {"GetSystemTimeAsFileTime", hook_GetSystemTimeAsFileTime},
    {"InitializeCriticalSection", hook_InitializeCriticalSection},
    {"EnterCriticalSection", hook_EnterCriticalSection},
    {"LeaveCriticalSection", hook_LeaveCriticalSection},
    {"DeleteCriticalSection", hook_DeleteCriticalSection},
    {"VirtualAlloc", hook_VirtualAlloc},
    {"VirtualFree", hook_VirtualFree},
    {"TlsAlloc", hook_TlsAlloc},
    {"TlsGetValue", hook_TlsGetValue},
    {"TlsSetValue", hook_TlsSetValue},
    {"TlsFree", hook_TlsFree},
    {"GetProcAddress", hook_GetProcAddress},
    {NULL, NULL}
};

void* resolve_symbol(const char* name) {
    for (int i = 0; g_symbols[i].name != NULL; i++) {
        if (strcmp(g_symbols[i].name, name) == 0) return g_symbols[i].func;
    }
    return NULL;
}

MS_ABI uint64_t generic_stub(void) { return 0; }

int main(int argc, char** argv) {
    FILE* f = fopen("target_app.exe", "rb");
    if (!f) return 1;
    fseek(f, 0, SEEK_END);
    size_t size = ftell(f);
    fseek(f, 0, SEEK_SET);
    uint8_t* pe_data = malloc(size);
    fread(pe_data, 1, size, f);
    fclose(f);

    uint32_t e_lfanew = *(uint32_t*)(pe_data + 0x3C);
    uint16_t num_sections = *(uint16_t*)(pe_data + e_lfanew + 0x6);
    uint8_t* opt_header = pe_data + e_lfanew + 0x18;

    uint64_t pref_base = *(uint64_t*)(opt_header + 0x18);
    uint32_t entry_rva = *(uint32_t*)(opt_header + 0x10);
    uint32_t image_size = *(uint32_t*)(opt_header + 0x38);
    uint32_t headers_size = *(uint32_t*)(opt_header + 0x3C);
    uint8_t* data_dirs = opt_header + 0x70;

    uint32_t import_rva = *(uint32_t*)(data_dirs + 0x8);
    uint32_t reloc_rva = *(uint32_t*)(data_dirs + 0x28);
    uint32_t reloc_size = *(uint32_t*)(data_dirs + 0x2C);

    g_image_base = (uint8_t*)mmap((void*)pref_base, image_size, PROT_READ | PROT_WRITE | PROT_EXEC, MAP_PRIVATE | MAP_ANONYMOUS, -1, 0);
    if (g_image_base == MAP_FAILED) {
        g_image_base = (uint8_t*)mmap(NULL, image_size, PROT_READ | PROT_WRITE | PROT_EXEC, MAP_PRIVATE | MAP_ANONYMOUS, -1, 0);
    }
    printf("[+] Target Image Mapped at: %p (Preferable: %p)\\n", g_image_base, (void*)pref_base);

    memcpy(g_image_base, pe_data, headers_size);

    uint8_t* sec_header = opt_header + *(uint16_t*)(pe_data + e_lfanew + 0x14);
    for (int i = 0; i < num_sections; i++) {
        uint8_t* sec = sec_header + i * 40;
        uint32_t v_addr = *(uint32_t*)(sec + 12);
        uint32_t raw_size = *(uint32_t*)(sec + 16);
        uint32_t raw_ptr = *(uint32_t*)(sec + 20);
        if (raw_size > 0) {
            memcpy(g_image_base + v_addr, pe_data + raw_ptr, raw_size);
        }
    }

    // Релокации
    int64_t delta = (int64_t)g_image_base - (int64_t)pref_base;
    if (reloc_rva && delta != 0) {
        uint32_t offset = 0;
        while (offset < reloc_size) {
            uint32_t page_rva = *(uint32_t*)(pe_data + reloc_rva + offset);
            uint32_t block_size = *(uint32_t*)(pe_data + reloc_rva + offset + 4);
            if (!block_size) break;
            int count = (block_size - 8) / 2;
            uint16_t* entries = (uint16_t*)(pe_data + reloc_rva + offset + 8);
            for (int j = 0; j < count; j++) {
                if ((entries[j] >> 12) == 10) {
                    uint64_t* target = (uint64_t*)(g_image_base + page_rva + (entries[j] & 0xFFF));
                    *target += delta;
                }
            }
            offset += block_size;
        }
    }

    // Разрешение IAT
    if (import_rva) {
        uint8_t* desc = g_image_base + import_rva;
        while (1) {
            uint32_t orig_thunk = *(uint32_t*)desc;
            uint32_t first_thunk = *(uint32_t*)(desc + 16);
            if (!orig_thunk && !first_thunk) break;
            uint32_t thunk_rva = orig_thunk ? orig_thunk : first_thunk;
            uint64_t* thunk = (uint64_t*)(g_image_base + thunk_rva);
            uint64_t* iat = (uint64_t*)(g_image_base + first_thunk);

            while (*thunk) {
                if (!(*thunk & (1ULL << 63))) {
                    char* name = (char*)(g_image_base + *thunk + 2);
                    void* resolved = resolve_symbol(name);
                    if (resolved) {
                        *iat = (uint64_t)resolved;
                    } else {
                        *iat = (uint64_t)generic_stub;
                    }
                }
                thunk++;
                iat++;
            }
            desc += 20;
        }
    }

    // Инициализация TEB/PEB и переключение регистра GS
    memset(&g_teb, 0, sizeof(g_teb));
    memset(&g_peb, 0, sizeof(g_peb));
    g_teb.Self = &g_teb;
    g_teb.ProcessEnvironmentBlock = &g_peb;
    g_peb.ProcessHeap = (void*)0x50000;

    syscall(SYS_arch_prctl, ARCH_SET_GS, &g_teb);
    printf("[+] Thread Environment Block (TEB) mapped to GS segment register.\\n");

    // Запуск
    typedef MS_ABI void (*WinEntry_t)(void);
    WinEntry_t entry = (WinEntry_t)(g_image_base + entry_rva);

    printf("[*] Invoking Windows C++ PE EntryPoint at %p...\\n", entry);

    if (setjmp(g_exit_env) == 0) {
        entry();
    }

    printf("\\n[*] Execution finished cleanly! ReturnCode: %d\\n", g_exit_code);
    return 0;
}
"""

with open("win_loader.c", "w") as f:
    f.write(loader_source)

# Компилируем нативный Linux лоадер
subprocess.run(["gcc", "-O2", "win_loader.c", "-o", "win_loader"], check=True)

# Запускаем бинарник Windows в Linux
res = subprocess.run(["./win_loader"], capture_output=True, text=True)
print(res.stdout)
if res.stderr:
    print("STDERR:", res.stderr)

[+] Target Windows C++ PE binary 'target_app.exe' successfully compiled.



In [4]:
import subprocess
import os

# 1. Скачиваем официальный релизный Windows PE x64 бинарник jq.exe с GitHub
url = "https://github.com/jqlang/jq/releases/download/jq-1.7.1/jq-windows-amd64.exe"
print(f"[*] Downloading official Windows PE binary from: {url}")
subprocess.run(["curl", "-L", "-o", "target_app.exe", url], check=True)

# Проверяем, что это действительно Windows PE x64
res_file = subprocess.run(["file", "target_app.exe"], capture_output=True, text=True)
print(f"[+] File signature: {res_file.stdout.strip()}")

# 2. Пишем полноценный C-лоадер и транслятор API
loader_c = """
#define _GNU_SOURCE
#include <stdio.h>
#include <stdlib.h>
#include <stdint.h>
#include <string.h>
#include <sys/mman.h>
#include <unistd.h>
#include <setjmp.h>
#include <sys/syscall.h>
#include <asm/prctl.h>
#include <wchar.h>

static jmp_buf g_exit_env;
static int g_exit_code = 0;
static uint8_t* g_image_base = NULL;

// --- TEB / PEB для Windows x64 (GS) ---
typedef struct {
    uint8_t Reserved[24];
    void* ProcessHeap;
} WIN_PEB;

typedef struct {
    void* ExceptionList;
    void* StackBase;
    void* StackLimit;
    void* SubSystemTib;
    void* FiberData;
    void* ArbitraryUserPointer;
    void* Self;                  // 0x30: Адрес самого TEB
    void* EnvironmentPointer;
    uint64_t ProcessId;
    uint64_t ThreadId;
    void* RpcHandle;
    void* Tls;
    WIN_PEB* ProcessEnvironmentBlock; // 0x60: Указатель на PEB
} __attribute__((packed)) WIN_TEB;

static WIN_TEB g_teb;
static WIN_PEB g_peb;

#define MS_ABI __attribute__((ms_abi))

// --- СИСТЕМНЫЕ ХУКИ KERNEL32 / CRT ---
MS_ABI void* hook_GetModuleHandleA(const char* name) { return g_image_base; }
MS_ABI void* hook_GetModuleHandleW(const wchar_t* name) { return g_image_base; }
MS_ABI void* hook_GetStdHandle(uint32_t nStdHandle) {
    if (nStdHandle == (uint32_t)-11) return (void*)1; // STDOUT
    if (nStdHandle == (uint32_t)-12) return (void*)2; // STDERR
    return (void*)0;                                  // STDIN
}

MS_ABI char* hook_GetCommandLineA(void) {
    static char cmd[] = "jq.exe --version";
    return cmd;
}

MS_ABI wchar_t* hook_GetCommandLineW(void) {
    static wchar_t cmd[] = L"jq.exe --version";
    return cmd;
}

MS_ABI void* hook_GetProcessHeap(void) { return (void*)0x50000; }
MS_ABI void* hook_HeapAlloc(void* h, uint32_t f, size_t sz) { return malloc(sz); }
MS_ABI int hook_HeapFree(void* h, uint32_t f, void* p) { free(p); return 1; }
MS_ABI void* hook_HeapReAlloc(void* h, uint32_t f, void* p, size_t sz) { return realloc(p, sz); }

MS_ABI int hook_WriteFile(void* hFile, const void* lpBuffer, uint32_t nSize, uint32_t* lpWritten, void* lpOverlapped) {
    int fd = (int)(uintptr_t)hFile;
    if (fd != 1 && fd != 2) fd = 1;
    ssize_t res = write(fd, lpBuffer, nSize);
    if (lpWritten && res > 0) *lpWritten = (uint32_t)res;
    return 1;
}

MS_ABI int hook_ReadFile(void* hFile, void* lpBuffer, uint32_t nSize, uint32_t* lpRead, void* lpOverlapped) {
    if (lpRead) *lpRead = 0;
    return 0; // EOF
}

MS_ABI void hook_ExitProcess(uint32_t uExitCode) {
    g_exit_code = (int)uExitCode;
    longjmp(g_exit_env, 1);
}

MS_ABI uint32_t hook_GetFileType(void* hFile) { return 0x0002; } // FILE_TYPE_CHAR
MS_ABI int hook_GetConsoleMode(void* hConsole, uint32_t* lpMode) { if (lpMode) *lpMode = 7; return 1; }
MS_ABI int hook_SetConsoleMode(void* hConsole, uint32_t dwMode) { return 1; }
MS_ABI uint32_t hook_GetLastError(void) { return 0; }
MS_ABI void hook_SetLastError(uint32_t e) {}
MS_ABI void* hook_SetUnhandledExceptionFilter(void* f) { return NULL; }
MS_ABI uint32_t hook_GetCurrentProcessId(void) { return getpid(); }
MS_ABI uint32_t hook_GetCurrentThreadId(void) { return getpid(); }
MS_ABI void hook_GetSystemTimeAsFileTime(void* p) { memset(p, 0, 8); }
MS_ABI void hook_InitializeCriticalSection(void* p) {}
MS_ABI void hook_EnterCriticalSection(void* p) {}
MS_ABI void hook_LeaveCriticalSection(void* p) {}
MS_ABI void hook_DeleteCriticalSection(void* p) {}
MS_ABI void* hook_VirtualAlloc(void* a, size_t sz, uint32_t t, uint32_t p) {
    return mmap(NULL, sz, PROT_READ | PROT_WRITE, MAP_PRIVATE | MAP_ANONYMOUS, -1, 0);
}
MS_ABI int hook_VirtualFree(void* a, size_t sz, uint32_t t) { munmap(a, sz); return 1; }
MS_ABI void* hook_TlsAlloc(void) { return (void*)1; }
MS_ABI void* hook_TlsGetValue(uint32_t i) { return NULL; }
MS_ABI int hook_TlsSetValue(uint32_t i, void* v) { return 1; }
MS_ABI int hook_TlsFree(uint32_t i) { return 1; }

MS_ABI int hook_MultiByteToWideChar(uint32_t cp, uint32_t flags, const char* src, int cbSrc, wchar_t* dst, int cchDst) {
    if (cbSrc == -1) cbSrc = strlen(src) + 1;
    if (cchDst == 0) return cbSrc;
    for (int i = 0; i < cbSrc && i < cchDst; i++) dst[i] = (wchar_t)src[i];
    return cbSrc;
}

MS_ABI int hook_WideCharToMultiByte(uint32_t cp, uint32_t flags, const wchar_t* src, int cchSrc, char* dst, int cbDst, const char* def, int* usedDef) {
    if (cchSrc == -1) cchSrc = wcslen(src) + 1;
    if (cbDst == 0) return cchSrc;
    for (int i = 0; i < cchSrc && i < cbDst; i++) dst[i] = (char)src[i];
    return cchSrc;
}

MS_ABI void* hook_GetProcAddress(void* hMod, const char* name);

// Таблица символов
typedef struct { const char* name; void* func; } SymbolEntry;
static SymbolEntry g_symbols[] = {
    {"GetModuleHandleA", hook_GetModuleHandleA},
    {"GetModuleHandleW", hook_GetModuleHandleW},
    {"GetStdHandle", hook_GetStdHandle},
    {"GetCommandLineA", hook_GetCommandLineA},
    {"GetCommandLineW", hook_GetCommandLineW},
    {"GetProcessHeap", hook_GetProcessHeap},
    {"HeapAlloc", hook_HeapAlloc},
    {"HeapFree", hook_HeapFree},
    {"HeapReAlloc", hook_HeapReAlloc},
    {"WriteFile", hook_WriteFile},
    {"ReadFile", hook_ReadFile},
    {"ExitProcess", hook_ExitProcess},
    {"GetFileType", hook_GetFileType},
    {"GetConsoleMode", hook_GetConsoleMode},
    {"SetConsoleMode", hook_SetConsoleMode},
    {"GetLastError", hook_GetLastError},
    {"SetLastError", hook_SetLastError},
    {"SetUnhandledExceptionFilter", hook_SetUnhandledExceptionFilter},
    {"GetCurrentProcessId", hook_GetCurrentProcessId},
    {"GetCurrentThreadId", hook_GetCurrentThreadId},
    {"GetSystemTimeAsFileTime", hook_GetSystemTimeAsFileTime},
    {"InitializeCriticalSection", hook_InitializeCriticalSection},
    {"EnterCriticalSection", hook_EnterCriticalSection},
    {"LeaveCriticalSection", hook_LeaveCriticalSection},
    {"DeleteCriticalSection", hook_DeleteCriticalSection},
    {"VirtualAlloc", hook_VirtualAlloc},
    {"VirtualFree", hook_VirtualFree},
    {"TlsAlloc", hook_TlsAlloc},
    {"TlsGetValue", hook_TlsGetValue},
    {"TlsSetValue", hook_TlsSetValue},
    {"TlsFree", hook_TlsFree},
    {"MultiByteToWideChar", hook_MultiByteToWideChar},
    {"WideCharToMultiByte", hook_WideCharToMultiByte},
    {"GetProcAddress", hook_GetProcAddress},
    {NULL, NULL}
};

void* resolve_symbol(const char* name) {
    for (int i = 0; g_symbols[i].name != NULL; i++) {
        if (strcmp(g_symbols[i].name, name) == 0) return g_symbols[i].func;
    }
    return NULL;
}

MS_ABI void* hook_GetProcAddress(void* hMod, const char* name) {
    return resolve_symbol(name);
}

MS_ABI uint64_t generic_stub(void) { return 0; }

// ====================================================================
// PE LOADER & RUNNER
// ====================================================================
int main(int argc, char** argv) {
    FILE* f = fopen("target_app.exe", "rb");
    if (!f) { printf("[-] Cannot open target_app.exe\\n"); return 1; }
    fseek(f, 0, SEEK_END);
    size_t size = ftell(f);
    fseek(f, 0, SEEK_SET);
    uint8_t* pe_data = malloc(size);
    fread(pe_data, 1, size, f);
    fclose(f);

    uint32_t e_lfanew = *(uint32_t*)(pe_data + 0x3C);
    uint16_t num_sections = *(uint16_t*)(pe_data + e_lfanew + 0x6);
    uint8_t* opt_header = pe_data + e_lfanew + 0x18;

    uint64_t pref_base = *(uint64_t*)(opt_header + 0x18);
    uint32_t entry_rva = *(uint32_t*)(opt_header + 0x10);
    uint32_t image_size = *(uint32_t*)(opt_header + 0x38);
    uint32_t headers_size = *(uint32_t*)(opt_header + 0x3C);
    uint8_t* data_dirs = opt_header + 0x70;

    uint32_t import_rva = *(uint32_t*)(data_dirs + 0x8);
    uint32_t reloc_rva = *(uint32_t*)(data_dirs + 0x28);
    uint32_t reloc_size = *(uint32_t*)(data_dirs + 0x2C);

    g_image_base = (uint8_t*)mmap((void*)pref_base, image_size, PROT_READ | PROT_WRITE | PROT_EXEC, MAP_PRIVATE | MAP_ANONYMOUS, -1, 0);
    if (g_image_base == MAP_FAILED) {
        g_image_base = (uint8_t*)mmap(NULL, image_size, PROT_READ | PROT_WRITE | PROT_EXEC, MAP_PRIVATE | MAP_ANONYMOUS, -1, 0);
    }
    printf("[+] Memory Base: %p (Original: %p)\\n", g_image_base, (void*)pref_base);

    memcpy(g_image_base, pe_data, headers_size);

    uint8_t* sec_header = opt_header + *(uint16_t*)(pe_data + e_lfanew + 0x14);
    for (int i = 0; i < num_sections; i++) {
        uint8_t* sec = sec_header + i * 40;
        uint32_t v_addr = *(uint32_t*)(sec + 12);
        uint32_t raw_size = *(uint32_t*)(sec + 16);
        uint32_t raw_ptr = *(uint32_t*)(sec + 20);
        if (raw_size > 0) {
            memcpy(g_image_base + v_addr, pe_data + raw_ptr, raw_size);
        }
    }

    // Релокации
    int64_t delta = (int64_t)g_image_base - (int64_t)pref_base;
    if (reloc_rva && delta != 0) {
        uint32_t offset = 0;
        while (offset < reloc_size) {
            uint32_t page_rva = *(uint32_t*)(pe_data + reloc_rva + offset);
            uint32_t block_size = *(uint32_t*)(pe_data + reloc_rva + offset + 4);
            if (!block_size) break;
            int count = (block_size - 8) / 2;
            uint16_t* entries = (uint16_t*)(pe_data + reloc_rva + offset + 8);
            for (int j = 0; j < count; j++) {
                if ((entries[j] >> 12) == 10) {
                    uint64_t* target = (uint64_t*)(g_image_base + page_rva + (entries[j] & 0xFFF));
                    *target += delta;
                }
            }
            offset += block_size;
        }
    }

    // Разрешение IAT
    if (import_rva) {
        uint8_t* desc = g_image_base + import_rva;
        while (1) {
            uint32_t orig_thunk = *(uint32_t*)desc;
            uint32_t first_thunk = *(uint32_t*)(desc + 16);
            if (!orig_thunk && !first_thunk) break;
            uint32_t thunk_rva = orig_thunk ? orig_thunk : first_thunk;
            uint64_t* thunk = (uint64_t*)(g_image_base + thunk_rva);
            uint64_t* iat = (uint64_t*)(g_image_base + first_thunk);

            while (*thunk) {
                if (!(*thunk & (1ULL << 63))) {
                    char* name = (char*)(g_image_base + *thunk + 2);
                    void* resolved = resolve_symbol(name);
                    *iat = resolved ? (uint64_t)resolved : (uint64_t)generic_stub;
                }
                thunk++;
                iat++;
            }
            desc += 20;
        }
    }

    // Инициализация TEB/PEB
    memset(&g_teb, 0, sizeof(g_teb));
    memset(&g_peb, 0, sizeof(g_peb));
    g_teb.Self = &g_teb;
    g_teb.ProcessEnvironmentBlock = &g_peb;
    g_peb.ProcessHeap = (void*)0x50000;

    syscall(SYS_arch_prctl, ARCH_SET_GS, &g_teb);

    typedef MS_ABI void (*WinEntry_t)(void);
    WinEntry_t entry = (WinEntry_t)(g_image_base + entry_rva);

    printf("[*] Executing Official Windows PE (jq.exe) at %p...\\n\\n", entry);

    if (setjmp(g_exit_env) == 0) {
        entry();
    }

    printf("\\n\\n[*] Official Windows PE Application executed and exited with code: %d\\n", g_exit_code);
    return 0;
}
"""

with open("real_win_loader.c", "w") as f:
    f.write(loader_c)

# Компилируем лоадер
subprocess.run(["gcc", "-O2", "real_win_loader.c", "-o", "real_win_loader"], check=True)

# Запускаем официальный Windows-бинарник в среде Linux
res = subprocess.run(["./real_win_loader"], capture_output=True, text=True)
print(res.stdout)
if res.stderr:
    print("STDERR:", res.stderr)

[*] Downloading official Windows PE binary from: https://github.com/jqlang/jq/releases/download/jq-1.7.1/jq-windows-amd64.exe
[+] File signature: target_app.exe: PE32+ executable (console) x86-64 (stripped to external PDB), for MS Windows



In [6]:
import subprocess
import os

# 1. Скачиваем официальный релизный Windows PE x64 бинарник UPX с GitHub через curl с User-Agent
url = "https://github.com/upx/upx/releases/download/v4.2.4/upx-4.2.4-win64.zip"
print(f"[*] Downloading official Windows PE binary from GitHub...")
subprocess.run(["curl", "-sL", "-A", "Mozilla/5.0", "-o", "upx.zip", url], check=True)
subprocess.run(["unzip", "-q", "-o", "upx.zip"], check=True)
subprocess.run(["cp", "upx-4.2.4-win64/upx.exe", "target_app.exe"], check=True)

res_file = subprocess.run(["file", "target_app.exe"], capture_output=True, text=True)
print(f"[+] Downloaded official binary: {res_file.stdout.strip()}")

# 2. Пишем лоадер с поддержкой Windows x64 ABI, TEB/PEB и перехватом CRT-вызовов
loader_c = """
#define _GNU_SOURCE
#include <stdio.h>
#include <stdlib.h>
#include <stdint.h>
#include <string.h>
#include <sys/mman.h>
#include <unistd.h>
#include <setjmp.h>
#include <sys/syscall.h>
#include <asm/prctl.h>
#include <wchar.h>

static jmp_buf g_exit_env;
static int g_exit_code = 0;
static uint8_t* g_image_base = NULL;

typedef struct {
    uint8_t Reserved[24];
    void* ProcessHeap;
} WIN_PEB;

typedef struct {
    void* ExceptionList;
    void* StackBase;
    void* StackLimit;
    void* SubSystemTib;
    void* FiberData;
    void* ArbitraryUserPointer;
    void* Self;                  // 0x30: TEB
    void* EnvironmentPointer;
    uint64_t ProcessId;
    uint64_t ThreadId;
    void* RpcHandle;
    void* Tls;
    WIN_PEB* ProcessEnvironmentBlock; // 0x60: PEB
} __attribute__((packed)) WIN_TEB;

static WIN_TEB g_teb;
static WIN_PEB g_peb;

#define MS_ABI __attribute__((ms_abi))

MS_ABI void* hook_GetModuleHandleA(const char* name) { return g_image_base; }
MS_ABI void* hook_GetModuleHandleW(const wchar_t* name) { return g_image_base; }
MS_ABI void* hook_GetStdHandle(uint32_t nStdHandle) {
    if (nStdHandle == (uint32_t)-11) return (void*)1;
    if (nStdHandle == (uint32_t)-12) return (void*)2;
    return (void*)0;
}

MS_ABI char* hook_GetCommandLineA(void) {
    static char cmd[] = "upx.exe --version";
    return cmd;
}

MS_ABI wchar_t* hook_GetCommandLineW(void) {
    static wchar_t cmd[] = L"upx.exe --version";
    return cmd;
}

MS_ABI void* hook_GetProcessHeap(void) { return (void*)0x50000; }
MS_ABI void* hook_HeapAlloc(void* h, uint32_t f, size_t sz) { return malloc(sz); }
MS_ABI int hook_HeapFree(void* h, uint32_t f, void* p) { free(p); return 1; }
MS_ABI void* hook_HeapReAlloc(void* h, uint32_t f, void* p, size_t sz) { return realloc(p, sz); }

MS_ABI int hook_WriteFile(void* hFile, const void* lpBuffer, uint32_t nSize, uint32_t* lpWritten, void* lpOverlapped) {
    int fd = (int)(uintptr_t)hFile;
    if (fd != 1 && fd != 2) fd = 1;
    ssize_t res = write(fd, lpBuffer, nSize);
    if (lpWritten && res > 0) *lpWritten = (uint32_t)res;
    return 1;
}

MS_ABI int hook_WriteConsoleA(void* hConsole, const char* lpBuffer, uint32_t nChars, uint32_t* lpWritten, void* lpReserved) {
    ssize_t res = write(STDOUT_FILENO, lpBuffer, nChars);
    if (lpWritten && res > 0) *lpWritten = (uint32_t)res;
    return 1;
}

MS_ABI int hook_WriteConsoleW(void* hConsole, const wchar_t* lpBuffer, uint32_t nChars, uint32_t* lpWritten, void* lpReserved) {
    char buf[4096];
    size_t len = wcstombs(buf, lpBuffer, sizeof(buf) - 1);
    if (len != (size_t)-1) write(STDOUT_FILENO, buf, len);
    if (lpWritten) *lpWritten = nChars;
    return 1;
}

MS_ABI void hook_ExitProcess(uint32_t uExitCode) {
    g_exit_code = (int)uExitCode;
    longjmp(g_exit_env, 1);
}

MS_ABI uint32_t hook_GetFileType(void* hFile) { return 0x0002; }
MS_ABI int hook_GetConsoleMode(void* hConsole, uint32_t* lpMode) { if (lpMode) *lpMode = 7; return 1; }
MS_ABI int hook_SetConsoleMode(void* hConsole, uint32_t dwMode) { return 1; }
MS_ABI uint32_t hook_GetLastError(void) { return 0; }
MS_ABI void hook_SetLastError(uint32_t e) {}
MS_ABI void* hook_SetUnhandledExceptionFilter(void* f) { return NULL; }
MS_ABI uint32_t hook_GetCurrentProcessId(void) { return getpid(); }
MS_ABI uint32_t hook_GetCurrentThreadId(void) { return getpid(); }
MS_ABI void hook_GetSystemTimeAsFileTime(void* p) { memset(p, 0, 8); }
MS_ABI void hook_InitializeCriticalSection(void* p) {}
MS_ABI void hook_EnterCriticalSection(void* p) {}
MS_ABI void hook_LeaveCriticalSection(void* p) {}
MS_ABI void hook_DeleteCriticalSection(void* p) {}

MS_ABI void* hook_VirtualAlloc(void* a, size_t sz, uint32_t t, uint32_t p) {
    return mmap(NULL, sz, PROT_READ | PROT_WRITE, MAP_PRIVATE | MAP_ANONYMOUS, -1, 0);
}
MS_ABI int hook_VirtualFree(void* a, size_t sz, uint32_t t) { munmap(a, sz); return 1; }
MS_ABI void* hook_TlsAlloc(void) { return (void*)1; }
MS_ABI void* hook_TlsGetValue(uint32_t i) { return NULL; }
MS_ABI int hook_TlsSetValue(uint32_t i, void* v) { return 1; }
MS_ABI int hook_TlsFree(uint32_t i) { return 1; }
MS_ABI void* hook_GetProcAddress(void* hMod, const char* name);

typedef struct { const char* name; void* func; } SymbolEntry;
static SymbolEntry g_symbols[] = {
    {"GetModuleHandleA", hook_GetModuleHandleA},
    {"GetModuleHandleW", hook_GetModuleHandleW},
    {"GetStdHandle", hook_GetStdHandle},
    {"GetCommandLineA", hook_GetCommandLineA},
    {"GetCommandLineW", hook_GetCommandLineW},
    {"GetProcessHeap", hook_GetProcessHeap},
    {"HeapAlloc", hook_HeapAlloc},
    {"HeapFree", hook_HeapFree},
    {"HeapReAlloc", hook_HeapReAlloc},
    {"WriteFile", hook_WriteFile},
    {"WriteConsoleA", hook_WriteConsoleA},
    {"WriteConsoleW", hook_WriteConsoleW},
    {"ExitProcess", hook_ExitProcess},
    {"GetFileType", hook_GetFileType},
    {"GetConsoleMode", hook_GetConsoleMode},
    {"SetConsoleMode", hook_SetConsoleMode},
    {"GetLastError", hook_GetLastError},
    {"SetLastError", hook_SetLastError},
    {"SetUnhandledExceptionFilter", hook_SetUnhandledExceptionFilter},
    {"GetCurrentProcessId", hook_GetCurrentProcessId},
    {"GetCurrentThreadId", hook_GetCurrentThreadId},
    {"GetSystemTimeAsFileTime", hook_GetSystemTimeAsFileTime},
    {"InitializeCriticalSection", hook_InitializeCriticalSection},
    {"EnterCriticalSection", hook_EnterCriticalSection},
    {"LeaveCriticalSection", hook_LeaveCriticalSection},
    {"DeleteCriticalSection", hook_DeleteCriticalSection},
    {"VirtualAlloc", hook_VirtualAlloc},
    {"VirtualFree", hook_VirtualFree},
    {"TlsAlloc", hook_TlsAlloc},
    {"TlsGetValue", hook_TlsGetValue},
    {"TlsSetValue", hook_TlsSetValue},
    {"TlsFree", hook_TlsFree},
    {"GetProcAddress", hook_GetProcAddress},
    {NULL, NULL}
};

void* resolve_symbol(const char* name) {
    for (int i = 0; g_symbols[i].name != NULL; i++) {
        if (strcmp(g_symbols[i].name, name) == 0) return g_symbols[i].func;
    }
    return NULL;
}

MS_ABI void* hook_GetProcAddress(void* hMod, const char* name) {
    return resolve_symbol(name);
}

MS_ABI uint64_t generic_stub(void) { return 0; }

int main(int argc, char** argv) {
    FILE* f = fopen("target_app.exe", "rb");
    if (!f) return 1;
    fseek(f, 0, SEEK_END);
    size_t size = ftell(f);
    fseek(f, 0, SEEK_SET);
    uint8_t* pe_data = malloc(size);
    fread(pe_data, 1, size, f);
    fclose(f);

    uint32_t e_lfanew = *(uint32_t*)(pe_data + 0x3C);
    uint16_t num_sections = *(uint16_t*)(pe_data + e_lfanew + 0x6);
    uint8_t* opt_header = pe_data + e_lfanew + 0x18;

    uint64_t pref_base = *(uint64_t*)(opt_header + 0x18);
    uint32_t entry_rva = *(uint32_t*)(opt_header + 0x10);
    uint32_t image_size = *(uint32_t*)(opt_header + 0x38);
    uint32_t headers_size = *(uint32_t*)(opt_header + 0x3C);
    uint8_t* data_dirs = opt_header + 0x70;

    uint32_t import_rva = *(uint32_t*)(data_dirs + 0x8);
    uint32_t reloc_rva = *(uint32_t*)(data_dirs + 0x28);
    uint32_t reloc_size = *(uint32_t*)(data_dirs + 0x2C);

    g_image_base = (uint8_t*)mmap((void*)pref_base, image_size, PROT_READ | PROT_WRITE | PROT_EXEC, MAP_PRIVATE | MAP_ANONYMOUS, -1, 0);
    if (g_image_base == MAP_FAILED) {
        g_image_base = (uint8_t*)mmap(NULL, image_size, PROT_READ | PROT_WRITE | PROT_EXEC, MAP_PRIVATE | MAP_ANONYMOUS, -1, 0);
    }
    printf("[+] Memory Base Allocated: %p (Original: %p)\\n", g_image_base, (void*)pref_base);

    memcpy(g_image_base, pe_data, headers_size);

    uint8_t* sec_header = opt_header + *(uint16_t*)(pe_data + e_lfanew + 0x14);
    for (int i = 0; i < num_sections; i++) {
        uint8_t* sec = sec_header + i * 40;
        uint32_t v_addr = *(uint32_t*)(sec + 12);
        uint32_t raw_size = *(uint32_t*)(sec + 16);
        uint32_t raw_ptr = *(uint32_t*)(sec + 20);
        if (raw_size > 0) {
            memcpy(g_image_base + v_addr, pe_data + raw_ptr, raw_size);
        }
    }

    int64_t delta = (int64_t)g_image_base - (int64_t)pref_base;
    if (reloc_rva && delta != 0) {
        uint32_t offset = 0;
        while (offset < reloc_size) {
            uint32_t page_rva = *(uint32_t*)(pe_data + reloc_rva + offset);
            uint32_t block_size = *(uint32_t*)(pe_data + reloc_rva + offset + 4);
            if (!block_size) break;
            int count = (block_size - 8) / 2;
            uint16_t* entries = (uint16_t*)(pe_data + reloc_rva + offset + 8);
            for (int j = 0; j < count; j++) {
                if ((entries[j] >> 12) == 10) {
                    uint64_t* target = (uint64_t*)(g_image_base + page_rva + (entries[j] & 0xFFF));
                    *target += delta;
                }
            }
            offset += block_size;
        }
    }

    if (import_rva) {
        uint8_t* desc = g_image_base + import_rva;
        while (1) {
            uint32_t orig_thunk = *(uint32_t*)desc;
            uint32_t first_thunk = *(uint32_t*)(desc + 16);
            if (!orig_thunk && !first_thunk) break;
            uint32_t thunk_rva = orig_thunk ? orig_thunk : first_thunk;
            uint64_t* thunk = (uint64_t*)(g_image_base + thunk_rva);
            uint64_t* iat = (uint64_t*)(g_image_base + first_thunk);

            while (*thunk) {
                if (!(*thunk & (1ULL << 63))) {
                    char* name = (char*)(g_image_base + *thunk + 2);
                    void* resolved = resolve_symbol(name);
                    *iat = resolved ? (uint64_t)resolved : (uint64_t)generic_stub;
                }
                thunk++;
                iat++;
            }
            desc += 20;
        }
    }

    memset(&g_teb, 0, sizeof(g_teb));
    memset(&g_peb, 0, sizeof(g_peb));
    g_teb.Self = &g_teb;
    g_teb.ProcessEnvironmentBlock = &g_peb;
    g_peb.ProcessHeap = (void*)0x50000;

    syscall(SYS_arch_prctl, ARCH_SET_GS, &g_teb);

    typedef MS_ABI void (*WinEntry_t)(void);
    WinEntry_t entry = (WinEntry_t)(g_image_base + entry_rva);

    printf("[*] Executing Official Windows PE (upx.exe) at %p...\\n\\n", entry);

    if (setjmp(g_exit_env) == 0) {
        entry();
    }

    printf("\\n[*] Application finished! ReturnCode: %d\\n", g_exit_code);
    return 0;
}
"""

with open("upx_loader.c", "w") as f:
    f.write(loader_c)

subprocess.run(["gcc", "-O2", "upx_loader.c", "-o", "upx_loader"], check=True)

# Запускаем лоадер
proc = subprocess.Popen(["./upx_loader"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
stdout, _ = proc.communicate()
print(stdout)

[*] Downloading official Windows PE binary from GitHub...
[+] Downloaded official binary: target_app.exe: PE32+ executable (console) x86-64 (stripped to external PDB), for MS Windows



In [7]:
import random
import copy
import operator

# 1. "Черный ящик" (Оригинальная функция, которую мы хотим реконструировать)
def target_blackbox(a, b, c):
    return (a * 2) + (b ^ c) - 5

# Тестовый набор векторов (Oracle Test Suite)
TEST_CASES = [
    ((a, b, c), target_blackbox(a, b, c))
    for a in range(-5, 6)
    for b in range(0, 4)
    for c in range(0, 4)
]

# 2. Набор базовых примитивов Lisp-рантайма
OPERATORS = {
    '+': (operator.add, 2),
    '-': (operator.sub, 2),
    '*': (operator.mul, 2),
    '^': (operator.xor, 2)
}
TERMINALS = ['arg0', 'arg1', 'arg2', 2, 5]

# 3. Интерпретатор S-выражений (Lisp Evaluator)
def eval_ast(ast, env):
    if isinstance(ast, str) and ast in env:
        return env[ast]
    if not isinstance(ast, list):
        return ast

    op = ast[0]
    func, arity = OPERATORS[op]
    args = [eval_ast(child, env) for child in ast[1:]]
    try:
        return func(*args)
    except Exception:
        return 0

# 4. Генератор случайных деревьев выражений
def random_tree(depth=3):
    if depth == 0 or (depth < 3 and random.random() < 0.3):
        return random.choice(TERMINALS)
    op = random.choice(list(OPERATORS.keys()))
    arity = OPERATORS[op][1]
    return [op] + [random_tree(depth - 1) for _ in range(arity)]

# 5. Fitness Function (Меньше = лучше)
def compute_fitness(ast):
    error = 0
    for (a, b, c), expected in TEST_CASES:
        env = {'arg0': a, 'arg1': b, 'arg2': c}
        actual = eval_ast(ast, env)
        error += abs(actual - expected)
    # Штраф за размер дерева (Parsimony Pressure)
    tree_size = len(str(ast))
    return error + (tree_size * 0.001)

# 6. Мутация дерева
def mutate(ast):
    ast_copy = copy.deepcopy(ast)
    if not isinstance(ast_copy, list) or random.random() < 0.4:
        return random_tree(random.randint(1, 2))

    # Мутируем одну из веток
    idx = random.randint(1, len(ast_copy) - 1)
    ast_copy[idx] = mutate(ast_copy[idx])
    return ast_copy

# 7. Генетический цикл
POPULATION_SIZE = 200
GENERATIONS = 100

population = [random_tree(random.randint(1, 3)) for _ in range(POPULATION_SIZE)]

print("[*] Запуск генетического синтеза Lisp-AST...")
best_overall = None
best_score = float('inf')

for gen in range(GENERATIONS):
    scored = [(compute_fitness(ind), ind) for ind in population]
    scored.sort(key=lambda x: x[0])

    current_best_score, current_best_ast = scored[0]
    if current_best_score < best_score:
        best_score = current_best_score
        best_overall = current_best_ast
        print(f"Gen {gen:02d} | Best Fitness: {best_score:.4f} | AST: {best_overall}")

    if best_score < 0.1: # Найдено точное соответствие
        print(f"\n[+] Идеальное AST-дерево синтезировано за {gen} поколений!")
        break

    # Селекция + Мутация
    survivors = [ind for _, ind in scored[:POPULATION_SIZE // 4]]
    new_pop = survivors[:]
    while len(new_pop) < POPULATION_SIZE:
        parent = random.choice(survivors)
        new_pop.append(mutate(parent))
    population = new_pop

print(f"\nРезультат: {best_overall}")

[*] Запуск генетического синтеза Lisp-AST...
Gen 00 | Best Fitness: 536.0160 | AST: ['-', 'arg0', 2]
Gen 02 | Best Fitness: 176.0410 | AST: ['-', ['+', 'arg0', 'arg0'], ['-', 5, 2]]
Gen 17 | Best Fitness: 176.0360 | AST: ['-', ['*', 2, 'arg0'], ['-', 5, 2]]
Gen 63 | Best Fitness: 0.0560 | AST: ['-', ['*', 2, 'arg0'], ['-', 5, ['^', 'arg1', 'arg2']]]

[+] Идеальное AST-дерево синтезировано за 63 поколений!

Результат: ['-', ['*', 2, 'arg0'], ['-', 5, ['^', 'arg1', 'arg2']]]


In [8]:
import pprint
from collections import defaultdict

# =====================================================================
# 1. ЧЕРНЫЙ ЯЩИК (Системный объект Windows со скрытым состоянием)
# =====================================================================
class WindowsMutexOracle:
    def __init__(self):
        self.state = "DESTROYED" # DESTROYED, UNOWNED, LOCKED

    def step(self, api_call):
        if api_call == "CreateMutex":
            if self.state == "DESTROYED":
                self.state = "UNOWNED"
                return (self.state, "STATUS_SUCCESS")
            return (self.state, "ERROR_ALREADY_EXISTS")

        elif api_call == "WaitForSingleObject":
            if self.state == "UNOWNED":
                self.state = "LOCKED"
                return (self.state, "WAIT_OBJECT_0")
            elif self.state == "LOCKED":
                return (self.state, "WAIT_TIMEOUT")
            return (self.state, "ERROR_INVALID_HANDLE")

        elif api_call == "ReleaseMutex":
            if self.state == "LOCKED":
                self.state = "UNOWNED"
                return (self.state, "STATUS_SUCCESS")
            return (self.state, "ERROR_NOT_OWNER")

        elif api_call == "CloseHandle":
            self.state = "DESTROYED"
            return (self.state, "STATUS_SUCCESS")

        return (self.state, "ERROR_INVALID_FUNCTION")


# =====================================================================
# 2. АВТОМАТИЧЕСКИЙ СИНТЕЗАТОР АВТОМАТА СОСТОЯНИЙ (FSM LEARNER)
# =====================================================================
class AutomatonLearner:
    def __init__(self, oracle_cls, alphabet):
        self.oracle_cls = oracle_cls
        self.alphabet = alphabet
        self.transitions = defaultdict(dict) # state -> {event: (next_state, out)}

    def explore(self, max_depth=3):
        # Активное исследование пространства состояний (BFS по дереву префиксов)
        queue = [[]] # Список последовательностей вызовов
        visited_states = set()

        while queue:
            seq = queue.pop(0)

            # Инициализируем свежий оракул и прогоняем префикс
            oracle = self.oracle_cls()
            curr_state = oracle.state
            for action in seq:
                curr_state, _ = oracle.step(action)

            visited_states.add(curr_state)

            # Проверяем все доступные переходы из текущего состояния
            for action in self.alphabet:
                oracle_test = self.oracle_cls()
                # Воспроизводим путь до текущего состояния
                for a in seq: oracle_test.step(a)

                next_state, output = oracle_test.step(action)

                if action not in self.transitions[curr_state]:
                    self.transitions[curr_state][action] = (next_state, output)
                    if next_state not in visited_states and len(seq) < max_depth:
                        queue.append(seq + [action])


# =====================================================================
# 3. ТРАНСЛЯЦИЯ АВТОМАТА В LISP S-EXPRESSION AST
# =====================================================================
def fsm_to_lisp_ast(transitions):
    ast = [":fsm/runtime", [":initial-state", "DESTROYED"]]
    state_blocks = []

    for state, actions in sorted(transitions.items()):
        handlers = []
        for event, (next_state, out_code) in sorted(actions.items()):
            # Каждое ребро графа -> Lisp-правило с сигналом и мутацией состояния
            handler = [
                ":on-event", event,
                [":seq",
                 [":res/mutate", ":state", next_state],
                 [":sig/return", out_code]]
            ]
            handlers.append(handler)

        state_block = [":state", state, handlers]
        state_blocks.append(state_block)

    ast.append([":state-table", state_blocks])
    return ast


# =====================================================================
# 4. ЗАПУСК И ПРОВЕРКА РЕЗУЛЬТАТА
# =====================================================================
API_ALPHABET = ["CreateMutex", "WaitForSingleObject", "ReleaseMutex", "CloseHandle"]

print("[*] 1. Извлечение автомата состояний из оракула Win32 API...")
learner = AutomatonLearner(WindowsMutexOracle, API_ALPHABET)
learner.explore()

print(f"[+] Автомат синтезирован. Найдено состояний: {len(learner.transitions)}")
for state, edges in learner.transitions.items():
    print(f"  Состояние [{state}]:")
    for ev, (nxt, out) in edges.items():
        print(f"    --({ev})--> [{nxt}] | Result: {out}")

print("\n[*] 2. Трансляция автомата в каноническое Lisp S-Expression AST:")
lisp_ast = fsm_to_lisp_ast(learner.transitions)

pprint.pprint(lisp_ast, depth=4, width=80)

[*] 1. Извлечение автомата состояний из оракула Win32 API...
[+] Автомат синтезирован. Найдено состояний: 3
  Состояние [DESTROYED]:
    --(CreateMutex)--> [UNOWNED] | Result: STATUS_SUCCESS
    --(WaitForSingleObject)--> [DESTROYED] | Result: ERROR_INVALID_HANDLE
    --(ReleaseMutex)--> [DESTROYED] | Result: ERROR_NOT_OWNER
    --(CloseHandle)--> [DESTROYED] | Result: STATUS_SUCCESS
  Состояние [UNOWNED]:
    --(CreateMutex)--> [UNOWNED] | Result: ERROR_ALREADY_EXISTS
    --(WaitForSingleObject)--> [LOCKED] | Result: WAIT_OBJECT_0
    --(ReleaseMutex)--> [UNOWNED] | Result: ERROR_NOT_OWNER
    --(CloseHandle)--> [DESTROYED] | Result: STATUS_SUCCESS
  Состояние [LOCKED]:
    --(CreateMutex)--> [LOCKED] | Result: ERROR_ALREADY_EXISTS
    --(WaitForSingleObject)--> [LOCKED] | Result: WAIT_TIMEOUT
    --(ReleaseMutex)--> [UNOWNED] | Result: STATUS_SUCCESS
    --(CloseHandle)--> [DESTROYED] | Result: STATUS_SUCCESS

[*] 2. Трансляция автомата в каноническое Lisp S-Expression AST:
[':fsm/ru

In [9]:
import subprocess
import ctypes

# 1. Генератор нативного C-кода из Lisp AST
def compile_lisp_ast_to_c(ast):
    initial_state = ast[1][1]
    states = ast[2][1]

    # Создаем enum состояний и событий
    state_names = [s[1] for s in states]

    c_source = f"""
#include <stdint.h>
#include <string.h>

typedef enum {{
"""
    for i, sname in enumerate(state_names):
        c_source += f"    STATE_{sname} = {i},\n"
    c_source += f"""}} MutexState;

static MutexState g_current_state = STATE_{initial_state};

__attribute__((ms_abi))
const char* dispatch_mutex_event(const char* event_name) {{
    switch (g_current_state) {{
"""
    for s_block in states:
        sname = s_block[1]
        handlers = s_block[2]
        c_source += f"    case STATE_{sname}:\n"
        for h in handlers:
            ev = h[1]
            next_state = h[2][1][2]
            ret_code = h[2][2][2]
            c_source += f'        if (strcmp(event_name, "{ev}") == 0) {{\n'
            c_source += f"            g_current_state = STATE_{next_state};\n"
            c_source += f'            return "{ret_code}";\n'
            c_source += f"        }}\n"
        c_source += "        break;\n\n"

    c_source += """    }
    return "ERROR_INVALID_STATE";
}

__attribute__((ms_abi))
int get_current_state_id() {
    return (int)g_current_state;
}
"""
    return c_source

# 2. Компилируем синтезированный Lisp-код в нативную библиотеку
c_code = compile_lisp_ast_to_c(lisp_ast)
with open("synthesized_runtime.c", "w") as f:
    f.write(c_code)

subprocess.run(["gcc", "-O3", "-shared", "-fPIC", "synthesized_runtime.c", "-o", "libsynthesized_runtime.so"], check=True)
print("[+] Lisp AST successfully compiled into native x86_64 machine code (.so)!")

# 3. Тестируем нативную скомпилированную машину состояний
lib = ctypes.CDLL("./libsynthesized_runtime.so")
lib.dispatch_mutex_event.argtypes = [ctypes.c_char_p]
lib.dispatch_mutex_event.restype = ctypes.c_char_p

def call_native(event):
    res = lib.dispatch_mutex_event(event.encode('ascii')).decode('ascii')
    state_id = lib.get_current_state_id()
    print(f"Native Dispatch -> Event: '{event:<20}' | Result: {res:<22} | New State ID: {state_id}")

print("\n--- Проверка работы нативного синтезированного рантайма ---")
call_native("CreateMutex")
call_native("WaitForSingleObject")
call_native("WaitForSingleObject")  # Должен вернуть TIMEOUT
call_native("ReleaseMutex")
call_native("CloseHandle")
call_native("WaitForSingleObject")  # Должен вернуть ERROR_INVALID_HANDLE

IndexError: list index out of range

In [10]:
import subprocess
import ctypes

# 1. Генератор нативного C-кода из Lisp AST
def compile_lisp_ast_to_c(ast):
    initial_state = ast[1][1]
    states = ast[2][1]

    # Список уникальных состояний
    state_names = [s[1] for s in states]

    c_source = """#include <stdint.h>
#include <string.h>

typedef enum {
"""
    for i, sname in enumerate(state_names):
        c_source += f"    STATE_{sname} = {i},\n"
    c_source += f"""}} MutexState;

static MutexState g_current_state = STATE_{initial_state};

__attribute__((ms_abi))
const char* dispatch_mutex_event(const char* event_name) {{
    switch (g_current_state) {{
"""
    for s_block in states:
        sname = s_block[1]
        handlers = s_block[2]
        c_source += f"    case STATE_{sname}:\n"
        for h in handlers:
            ev = h[1]
            seq_block = h[2] # [':seq', [':res/mutate', ':state', next_state], [':sig/return', out_code]]
            next_state = seq_block[1][2]
            ret_code = seq_block[2][1] # Исправленный индекс

            c_source += f'        if (strcmp(event_name, "{ev}") == 0) {{\n'
            c_source += f"            g_current_state = STATE_{next_state};\n"
            c_source += f'            return "{ret_code}";\n'
            c_source += f"        }}\n"
        c_source += "        break;\n\n"

    c_source += """    }
    return "ERROR_INVALID_STATE";
}

__attribute__((ms_abi))
int get_current_state_id() {
    return (int)g_current_state;
}
"""
    return c_source

# 2. Компилируем синтезированный Lisp-код в нативную библиотеку (.so)
c_code = compile_lisp_ast_to_c(lisp_ast)
with open("synthesized_runtime.c", "w") as f:
    f.write(c_code)

subprocess.run(["gcc", "-O3", "-shared", "-fPIC", "synthesized_runtime.c", "-o", "libsynthesized_runtime.so"], check=True)
print("[+] Lisp AST successfully compiled into native x86_64 machine code (.so)!\n")

# 3. Тестируем нативную скомпилированную машину состояний
lib = ctypes.CDLL("./libsynthesized_runtime.so")
lib.dispatch_mutex_event.argtypes = [ctypes.c_char_p]
lib.dispatch_mutex_event.restype = ctypes.c_char_p
lib.get_current_state_id.restype = ctypes.c_int

def call_native(event):
    res = lib.dispatch_mutex_event(event.encode('ascii')).decode('ascii')
    state_id = lib.get_current_state_id()
    print(f"Native Dispatch -> Event: '{event:<20}' | Result: {res:<22} | State ID: {state_id}")

print("--- Проверка работы скомпилированного нативного рантайма ---")
call_native("CreateMutex")
call_native("WaitForSingleObject")
call_native("WaitForSingleObject")  # Должен вернуть WAIT_TIMEOUT
call_native("ReleaseMutex")
call_native("CloseHandle")
call_native("WaitForSingleObject")  # Должен вернуть ERROR_INVALID_HANDLE

[+] Lisp AST successfully compiled into native x86_64 machine code (.so)!

--- Проверка работы скомпилированного нативного рантайма ---
Native Dispatch -> Event: 'CreateMutex         ' | Result: ERROR_INVALID_STATE    | State ID: 0
Native Dispatch -> Event: 'WaitForSingleObject ' | Result: ERROR_INVALID_STATE    | State ID: 0
Native Dispatch -> Event: 'WaitForSingleObject ' | Result: ERROR_INVALID_STATE    | State ID: 0
Native Dispatch -> Event: 'ReleaseMutex        ' | Result: ERROR_INVALID_STATE    | State ID: 0
Native Dispatch -> Event: 'CloseHandle         ' | Result: ERROR_INVALID_STATE    | State ID: 0
Native Dispatch -> Event: 'WaitForSingleObject ' | Result: ERROR_INVALID_STATE    | State ID: 0


In [13]:
import subprocess
import ctypes
from collections import defaultdict

# =====================================================================
# 1. ЧЕРНЫЙ ЯЩИК (Win32 Mutex Oracle)
# =====================================================================
class WindowsMutexOracle:
    def __init__(self):
        self.state = "DESTROYED"

    def step(self, api_call):
        if api_call == "CreateMutex":
            if self.state == "DESTROYED":
                self.state = "UNOWNED"
                return (self.state, "STATUS_SUCCESS")
            return (self.state, "ERROR_ALREADY_EXISTS")

        elif api_call == "WaitForSingleObject":
            if self.state == "UNOWNED":
                self.state = "LOCKED"
                return (self.state, "WAIT_OBJECT_0")
            elif self.state == "LOCKED":
                return (self.state, "WAIT_TIMEOUT")
            return (self.state, "ERROR_INVALID_HANDLE")

        elif api_call == "ReleaseMutex":
            if self.state == "LOCKED":
                self.state = "UNOWNED"
                return (self.state, "STATUS_SUCCESS")
            return (self.state, "ERROR_NOT_OWNER")

        elif api_call == "CloseHandle":
            self.state = "DESTROYED"
            return (self.state, "STATUS_SUCCESS")

        return (self.state, "ERROR_INVALID_FUNCTION")


# =====================================================================
# 2. АКТИВНОЕ ИЗВЛЕЧЕНИЕ АВТОМАТА (FSM Learner)
# =====================================================================
API_ALPHABET = ["CreateMutex", "WaitForSingleObject", "ReleaseMutex", "CloseHandle"]
transitions = defaultdict(dict)

queue = [[]]
visited_states = set()

while queue:
    seq = queue.pop(0)
    oracle = WindowsMutexOracle()
    curr_state = oracle.state
    for a in seq:
        curr_state, _ = oracle.step(a)
    visited_states.add(curr_state)

    for action in API_ALPHABET:
        oracle_test = WindowsMutexOracle()
        for a in seq: oracle_test.step(a)
        next_state, out = oracle_test.step(action)

        if action not in transitions[curr_state]:
            transitions[curr_state][action] = (next_state, out)
            if next_state not in visited_states and len(seq) < 3:
                queue.append(seq + [action])


# =====================================================================
# 3. ПОСТРОЕНИЕ LISP AST
# =====================================================================
lisp_ast = [":fsm/runtime", [":initial-state", "DESTROYED"], [":state-table", []]]
for state, actions in sorted(transitions.items()):
    handlers = []
    for ev, (nxt, out) in sorted(actions.items()):
        handlers.append([":on-event", ev, [":seq", [":res/mutate", ":state", nxt], [":sig/return", out]]])
    lisp_ast[2][1].append([":state", state, handlers])


# =====================================================================
# 4. ГЕНЕРАЦИЯ СТАНДАРТНОГО C-КОДА (System V ABI для вызова из Python)
# =====================================================================
initial_state = lisp_ast[1][1]
states = lisp_ast[2][1]
state_names = [s[1] for s in states]

c_code = """#include <stdint.h>
#include <string.h>

typedef enum {
"""
for i, sname in enumerate(state_names):
    c_code += f"    STATE_{sname} = {i},\n"
c_code += f"""}} MutexState;

static MutexState g_current_state = STATE_{initial_state};

const char* dispatch_mutex_event(const char* event_name) {{
    switch (g_current_state) {{
"""

for s_block in states:
    sname = s_block[1]
    handlers = s_block[2]
    c_code += f"    case STATE_{sname}:\n"
    for h in handlers:
        ev = h[1]
        nxt = h[2][1][2]
        ret = h[2][2][1]
        c_code += f'        if (strcmp(event_name, "{ev}") == 0) {{\n'
        c_code += f"            g_current_state = STATE_{nxt};\n"
        c_code += f'            return "{ret}";\n'
        c_code += f"        }}\n"
    c_code += "        break;\n\n"

c_code += """    }
    return "ERROR_INVALID_STATE";
}

int get_current_state_id(void) {
    return (int)g_current_state;
}
"""

with open("synthesized_runtime.c", "w") as f:
    f.write(c_code)

# Компилируем .so
subprocess.run(["gcc", "-O3", "-shared", "-fPIC", "synthesized_runtime.c", "-o", "libsynthesized_runtime.so"], check=True)

# =====================================================================
# 5. ТЕСТ СКОМПИЛИРОВАННОЙ БИБЛИОТЕКИ
# =====================================================================
lib = ctypes.CDLL("./libsynthesized_runtime.so")
lib.dispatch_mutex_event.argtypes = [ctypes.c_char_p]
lib.dispatch_mutex_event.restype = ctypes.c_char_p
lib.get_current_state_id.restype = ctypes.c_int

def call_native(event):
    res = lib.dispatch_mutex_event(event.encode('ascii')).decode('ascii')
    state_id = lib.get_current_state_id()
    print(f"Native Dispatch -> Event: '{event:<20}' | Result: {res:<22} | State ID: {state_id}")

print("--- Проверка работы скомпилированного нативного автомата ---")
call_native("CreateMutex")
call_native("WaitForSingleObject")
call_native("WaitForSingleObject")
call_native("ReleaseMutex")
call_native("CloseHandle")
call_native("WaitForSingleObject")

--- Проверка работы скомпилированного нативного автомата ---
Native Dispatch -> Event: 'CreateMutex         ' | Result: ERROR_INVALID_STATE    | State ID: 0
Native Dispatch -> Event: 'WaitForSingleObject ' | Result: ERROR_INVALID_STATE    | State ID: 0
Native Dispatch -> Event: 'WaitForSingleObject ' | Result: ERROR_INVALID_STATE    | State ID: 0
Native Dispatch -> Event: 'ReleaseMutex        ' | Result: ERROR_INVALID_STATE    | State ID: 0
Native Dispatch -> Event: 'CloseHandle         ' | Result: ERROR_INVALID_STATE    | State ID: 0
Native Dispatch -> Event: 'WaitForSingleObject ' | Result: ERROR_INVALID_STATE    | State ID: 0


In [14]:
import subprocess
import os

# 1. Устанавливаем MinGW для сборки Windows PE x64
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "g++-mingw-w64-x86-64"], check=True)

# 2. Исходный код Windows C++ DirectX приложения
win_d3d_cpp = """
#include <windows.h>
#include <stdint.h>

// Определение COM VTable для Direct3D9
struct IDirect3DDevice9;

struct IDirect3D9Vtbl {
    void* QueryInterface;
    void* AddRef;
    void* Release;
    void* RegisterSoftwareDevice;
    void* GetAdapterCount;
    void* GetAdapterIdentifier;
    void* GetAdapterModeCount;
    void* EnumAdapterModes;
    void* GetAdapterDisplayMode;
    void* CheckDeviceType;
    void* CheckDeviceFormat;
    void* CheckDeviceMultiSampleType;
    void* CheckDepthStencilMatch;
    void* CheckDeviceFormatConversion;
    void* GetDeviceCaps;
    void* GetAdapterMonitor;
    HRESULT (__attribute__((ms_abi)) *CreateDevice)(void* This, UINT Adapter, DWORD DeviceType, HWND hFocusWindow, DWORD BehaviorFlags, void* pPresentationParameters, IDirect3DDevice9** ppReturnedDeviceInterface);
};

struct IDirect3D9 {
    struct IDirect3D9Vtbl* lpVtbl;
};

struct Vertex {
    float x, y, z;
    uint32_t color; // 0xAARRGGBB
};

struct IDirect3DDevice9Vtbl {
    void* QueryInterface;
    void* AddRef;
    void* Release;
    void* TestCooperativeLevel;
    void* GetAvailableTextureMem;
    void* EvictManagedResources;
    void* GetDirect3D;
    void* GetDeviceCaps;
    void* GetDisplayMode;
    void* GetCreationParameters;
    void* SetCursorProperties;
    void* SetCursorPosition;
    void* ShowCursor;
    void* CreateAdditionalSwapChain;
    void* GetSwapChain;
    void* GetNumberOfSwapChains;
    void* Reset;
    HRESULT (__attribute__((ms_abi)) *Present)(IDirect3DDevice9* This, const void* pSourceRect, const void* pDestRect, HWND hDestWindowOverride, const void* pDirtyRegion);
    void* GetBackBuffer;
    void* GetRasterStatus;
    void* SetDialogBoxMode;
    void* SetGammaRamp;
    void* GetGammaRamp;
    void* CreateTexture;
    void* CreateVolumeTexture;
    void* CreateCubeTexture;
    void* CreateVertexBuffer;
    void* CreateIndexBuffer;
    void* CreateRenderTarget;
    void* CreateDepthStencilSurface;
    void* UpdateSurface;
    void* UpdateTexture;
    void* GetRenderTargetData;
    void* GetFrontBufferData;
    void* StretchRect;
    void* ColorFill;
    void* CreateOffscreenPlainSurface;
    void* SetRenderTarget;
    void* GetRenderTarget;
    void* SetDepthStencilSurface;
    void* GetDepthStencilSurface;
    HRESULT (__attribute__((ms_abi)) *BeginScene)(IDirect3DDevice9* This);
    HRESULT (__attribute__((ms_abi)) *EndScene)(IDirect3DDevice9* This);
    HRESULT (__attribute__((ms_abi)) *Clear)(IDirect3DDevice9* This, DWORD Count, const void* pRects, DWORD Flags, uint32_t Color, float Z, DWORD Stencil);
    void* SetTransform;
    void* GetTransform;
    void* MultiplyTransform;
    void* SetViewport;
    void* GetViewport;
    void* SetMaterial;
    void* GetMaterial;
    void* SetLight;
    void* GetLight;
    void* LightEnable;
    void* GetLightEnable;
    void* SetClipPlane;
    void* GetClipPlane;
    void* SetRenderState;
    void* GetRenderState;
    void* CreateStateBlock;
    void* BeginStateBlock;
    void* EndStateBlock;
    void* SetClipStatus;
    void* GetClipStatus;
    void* GetTexture;
    void* SetTexture;
    void* GetTextureStageState;
    void* SetTextureStageState;
    void* GetSamplerState;
    void* SetSamplerState;
    void* ValidateDevice;
    void* SetPaletteEntries;
    void* GetPaletteEntries;
    void* SetCurrentTexturePalette;
    void* GetCurrentTexturePalette;
    void* ScissorRect;
    void* SetSoftwareVertexProcessing;
    void* GetSoftwareVertexProcessing;
    void* SetNPatchMode;
    void* GetNPatchMode;
    HRESULT (__attribute__((ms_abi)) *DrawPrimitiveUP)(IDirect3DDevice9* This, DWORD PrimitiveType, UINT PrimitiveCount, const void* pVertexStreamZeroData, UINT VertexStreamZeroStride);
};

struct IDirect3DDevice9 {
    struct IDirect3DDevice9Vtbl* lpVtbl;
};

// Функция импорта из d3d9.dll
extern "C" __declspec(dllimport) IDirect3D9* __attribute__((ms_abi)) Direct3DCreate9(UINT SDKVersion);

int main() {
    // 1. Инициализация Direct3D9
    IDirect3D9* d3d = Direct3DCreate9(32);
    if (!d3d) return 1;

    IDirect3DDevice9* device = NULL;
    d3d->lpVtbl->CreateDevice(d3d, 0, 1, NULL, 0, NULL, &device);
    if (!device) return 2;

    // 2. Очистка буфера (Темно-синий фон 0xFF101828)
    device->lpVtbl->Clear(device, 0, NULL, 1, 0xFF101828, 1.0f, 0);

    // 3. Вершины цветного треугольника
    Vertex tri[3] = {
        { 40.0f,  8.0f, 0.0f, 0xFFFF2020 }, // Красная вершина (Top)
        { 70.0f, 32.0f, 0.0f, 0xFF20FF20 }, // Зеленая вершина (Right)
        { 10.0f, 32.0f, 0.0f, 0xFF2060FF }  // Синяя вершина (Left)
    };

    // 4. Пайплайн рендеринга
    device->lpVtbl->BeginScene(device);
    device->lpVtbl->DrawPrimitiveUP(device, 4 /* D3DPT_TRIANGLELIST */, 1, tri, sizeof(Vertex));
    device->lpVtbl->EndScene(device);

    // 5. Вывод кадра
    device->lpVtbl->Present(device, NULL, NULL, NULL, NULL);

    ExitProcess(0);
    return 0;
}
"""

with open("d3d_app.cpp", "w") as f:
    f.write(win_d3d_cpp)

# Собираем реальный Windows PE x64 бинарник с линковкой против фиктивной d3d9 библиотеки
with open("d3d9.def", "w") as f:
    f.write("LIBRARY d3d9.dll\nEXPORTS\nDirect3DCreate9\n")

subprocess.run(["x86_64-w64-mingw32-dlltool", "-d", "d3d9.def", "-l", "libd3d9.a"], check=True)
subprocess.run([
    "x86_64-w64-mingw32-g++",
    "-O2",
    "-static-libgcc", "-static-libstdc++",
    "d3d_app.cpp",
    "-o", "d3d_app.exe",
    "-L.", "-ld3d9", "-lkernel32"
], check=True)

print("[+] Windows Direct3D PE binary 'd3d_app.exe' compiled successfully.")

# ====================================================================
# 3. LINUX PE-ЛОАДЕР И СИНТЕЗИРОВАННЫЙ RASTERIZER РАHТАЙМ DIRECTX
# ====================================================================
loader_d3d_c = """
#define _GNU_SOURCE
#include <stdio.h>
#include <stdlib.h>
#include <stdint.h>
#include <string.h>
#include <sys/mman.h>
#include <unistd.h>
#include <setjmp.h>
#include <sys/syscall.h>
#include <asm/prctl.h>
#include <math.h>

#define MS_ABI __attribute__((ms_abi))

#define FB_WIDTH 80
#define FB_HEIGHT 38

static uint32_t g_framebuffer[FB_HEIGHT][FB_WIDTH];
static jmp_buf g_exit_env;
static uint8_t* g_image_base = NULL;

// TEB/PEB
typedef struct { uint8_t r[24]; void* ProcessHeap; } WIN_PEB;
typedef struct {
    void* r1[6]; void* Self; void* r2[4]; void* Tls; WIN_PEB* Peb;
} __attribute__((packed)) WIN_TEB;

static WIN_TEB g_teb;
static WIN_PEB g_peb;

// --- COM СИНТЕЗИРОВАННЫЙ DIRECT3D ДВИЖОК ---
struct IDirect3DDevice9;
struct IDirect3D9;

typedef struct {
    float x, y, z;
    uint32_t color;
} D3DVertex;

// Программный растеризатор треугольников (Barycentric Coordinates)
static void draw_triangle(const D3DVertex* v0, const D3DVertex* v1, const D3DVertex* v2) {
    int min_x = (int)fmaxf(0, fminf(v0->x, fminf(v1->x, v2->x)));
    int max_x = (int)fminf(FB_WIDTH - 1, fmaxf(v0->x, fmaxf(v1->x, v2->x)));
    int min_y = (int)fmaxf(0, fminf(v0->y, fminf(v1->y, v2->y)));
    int max_y = (int)fminf(FB_HEIGHT - 1, fmaxf(v0->y, fmaxf(v1->y, v2->y)));

    float denom = (v1->y - v2->y) * (v0->x - v2->x) + (v2->x - v1->x) * (v0->y - v2->y);
    if (fabsf(denom) < 1e-5f) return;

    for (int y = min_y; y <= max_y; y++) {
        for (int x = min_x; x <= max_x; x++) {
            float px = (float)x + 0.5f;
            float py = (float)y + 0.5f;

            float w0 = ((v1->y - v2->y) * (px - v2->x) + (v2->x - v1->x) * (py - v2->y)) / denom;
            float w1 = ((v2->y - v0->y) * (px - v2->x) + (v0->x - v2->x) * (py - v2->y)) / denom;
            float w2 = 1.0f - w0 - w1;

            if (w0 >= 0 && w1 >= 0 && w2 >= 0) {
                // Интерполяция цветов вершин RGB
                uint8_t r = (uint8_t)(w0 * ((v0->color >> 16) & 0xFF) + w1 * ((v1->color >> 16) & 0xFF) + w2 * ((v2->color >> 16) & 0xFF));
                uint8_t g = (uint8_t)(w0 * ((v0->color >> 8) & 0xFF)  + w1 * ((v1->color >> 8) & 0xFF)  + w2 * ((v2->color >> 8) & 0xFF));
                uint8_t b = (uint8_t)(w0 * (v0->color & 0xFF)         + w1 * (v1->color & 0xFF)         + w2 * (v2->color & 0xFF));
                g_framebuffer[y][x] = (0xFF << 24) | (r << 16) | (g << 8) | b;
            }
        }
    }
}

// COM Методы IDirect3DDevice9
MS_ABI int dev_Clear(void* This, uint32_t Count, const void* pRects, uint32_t Flags, uint32_t Color, float Z, uint32_t Stencil) {
    for (int y = 0; y < FB_HEIGHT; y++) {
        for (int x = 0; x < FB_WIDTH; x++) {
            g_framebuffer[y][x] = Color;
        }
    }
    return 0; // D3D_OK
}

MS_ABI int dev_BeginScene(void* This) { return 0; }
MS_ABI int dev_EndScene(void* This) { return 0; }

MS_ABI int dev_DrawPrimitiveUP(void* This, uint32_t PrimType, uint32_t PrimCount, const void* pData, uint32_t Stride) {
    const D3DVertex* vertices = (const D3DVertex*)pData;
    for (uint32_t i = 0; i < PrimCount; i++) {
        draw_triangle(&vertices[i * 3], &vertices[i * 3 + 1], &vertices[i * 3 + 2]);
    }
    return 0;
}

MS_ABI int dev_Present(void* This, const void* src, const void* dst, void* wnd, const void* dirty) {
    printf("\\n>>> [DirectX 9 Present Intercepted - Framebuffer Rendered in Linux]:\\n\\n");
    // Отрисовка ANSI TrueColor в терминале
    for (int y = 0; y < FB_HEIGHT; y += 2) {
        for (int x = 0; x < FB_WIDTH; x++) {
            uint32_t top = g_framebuffer[y][x];
            uint32_t bot = (y + 1 < FB_HEIGHT) ? g_framebuffer[y + 1][x] : top;

            uint8_t tr = (top >> 16) & 0xFF, tg = (top >> 8) & 0xFF, tb = top & 0xFF;
            uint8_t br = (bot >> 16) & 0xFF, bg = (bot >> 8) & 0xFF, bb = bot & 0xFF;

            // Символ полублока с TrueColor фоном и передним планом
            printf("\\033[38;2;%d;%d;%dm\\033[48;2;%d;%d;%dm▀", tr, tg, tb, br, bg, bb);
        }
        printf("\\033[0m\\n");
    }
    printf("\\033[0m\\n");
    return 0;
}

static void* g_dev_vtbl[100];
static struct { void* vtbl; } g_device_instance = { g_dev_vtbl };

// COM Методы IDirect3D9
MS_ABI int d3d9_CreateDevice(void* This, uint32_t Adapter, uint32_t DevType, void* hWnd, uint32_t BehaviorFlags, void* pParams, void** ppDevice) {
    memset(g_dev_vtbl, 0, sizeof(g_dev_vtbl));
    g_dev_vtbl[17] = (void*)dev_Present;
    g_dev_vtbl[41] = (void*)dev_BeginScene;
    g_dev_vtbl[42] = (void*)dev_EndScene;
    g_dev_vtbl[43] = (void*)dev_Clear;
    g_dev_vtbl[82] = (void*)dev_DrawPrimitiveUP;

    *ppDevice = &g_device_instance;
    return 0; // D3D_OK
}

static void* g_d3d9_vtbl[30];
static struct { void* vtbl; } g_d3d9_instance = { g_d3d9_vtbl };

MS_ABI void* hook_Direct3DCreate9(uint32_t SDKVersion) {
    memset(g_d3d9_vtbl, 0, sizeof(g_d3d9_vtbl));
    g_d3d9_vtbl[16] = (void*)d3d9_CreateDevice;
    return &g_d3d9_instance;
}

// Стандартные Kernel32 заглушки
MS_ABI void hook_ExitProcess(uint32_t code) { longjmp(g_exit_env, 1); }
MS_ABI void* hook_GetModuleHandleA(const char* n) { return g_image_base; }
MS_ABI void* hook_GetProcessHeap(void) { return (void*)0x50000; }
MS_ABI void* hook_HeapAlloc(void* h, uint32_t f, size_t s) { return malloc(s); }
MS_ABI int hook_HeapFree(void* h, uint32_t f, void* p) { free(p); return 1; }
MS_ABI void* hook_SetUnhandledExceptionFilter(void* p) { return NULL; }

typedef struct { const char* name; void* func; } SymbolEntry;
static SymbolEntry g_symbols[] = {
    {"Direct3DCreate9", hook_Direct3DCreate9},
    {"ExitProcess", hook_ExitProcess},
    {"GetModuleHandleA", hook_GetModuleHandleA},
    {"GetProcessHeap", hook_GetProcessHeap},
    {"HeapAlloc", hook_HeapAlloc},
    {"HeapFree", hook_HeapFree},
    {"SetUnhandledExceptionFilter", hook_SetUnhandledExceptionFilter},
    {NULL, NULL}
};

void* resolve_sym(const char* name) {
    for (int i = 0; g_symbols[i].name; i++) {
        if (strcmp(g_symbols[i].name, name) == 0) return g_symbols[i].func;
    }
    return NULL;
}

MS_ABI uint64_t dummy_stub(void) { return 0; }

// ====================================================================
// LOADER
// ====================================================================
int main() {
    FILE* f = fopen("d3d_app.exe", "rb");
    if (!f) return 1;
    fseek(f, 0, SEEK_END);
    size_t size = ftell(f);
    fseek(f, 0, SEEK_SET);
    uint8_t* pe = malloc(size);
    fread(pe, 1, size, f);
    fclose(f);

    uint32_t e_lfanew = *(uint32_t*)(pe + 0x3C);
    uint16_t n_sec = *(uint16_t*)(pe + e_lfanew + 0x6);
    uint8_t* opt = pe + e_lfanew + 0x18;

    uint64_t pref_base = *(uint64_t*)(opt + 0x18);
    uint32_t entry_rva = *(uint32_t*)(opt + 0x10);
    uint32_t img_size = *(uint32_t*)(opt + 0x38);
    uint32_t hdr_size = *(uint32_t*)(opt + 0x3C);
    uint8_t* ddirs = opt + 0x70;

    uint32_t imp_rva = *(uint32_t*)(ddirs + 0x8);
    uint32_t rel_rva = *(uint32_t*)(ddirs + 0x28);
    uint32_t rel_sz = *(uint32_t*)(ddirs + 0x2C);

    g_image_base = (uint8_t*)mmap((void*)pref_base, img_size, PROT_READ | PROT_WRITE | PROT_EXEC, MAP_PRIVATE | MAP_ANONYMOUS, -1, 0);
    if (g_image_base == MAP_FAILED) {
        g_image_base = (uint8_t*)mmap(NULL, img_size, PROT_READ | PROT_WRITE | PROT_EXEC, MAP_PRIVATE | MAP_ANONYMOUS, -1, 0);
    }
    printf("[+] Direct3D App Mapped at: %p (Original Base: %p)\\n", g_image_base, (void*)pref_base);

    memcpy(g_image_base, pe, hdr_size);
    uint8_t* sec_hdr = opt + *(uint16_t*)(pe + e_lfanew + 0x14);
    for (int i = 0; i < n_sec; i++) {
        uint8_t* s = sec_hdr + i * 40;
        uint32_t va = *(uint32_t*)(s + 12);
        uint32_t vs = *(uint32_t*)(s + 8);
        uint32_t rs = *(uint32_t*)(s + 16);
        uint32_t rp = *(uint32_t*)(s + 20);
        if (rs > 0) memcpy(g_image_base + va, pe + rp, rs);
        if (vs > rs) memset(g_image_base + va + rs, 0, vs - rs);
    }

    // Релокации
    int64_t delta = (int64_t)g_image_base - (int64_t)pref_base;
    if (rel_rva && delta != 0) {
        uint32_t off = 0;
        while (off < rel_sz) {
            uint32_t pg = *(uint32_t*)(pe + rel_rva + off);
            uint32_t blk = *(uint32_t*)(pe + rel_rva + off + 4);
            if (!blk) break;
            int cnt = (blk - 8) / 2;
            uint16_t* ents = (uint16_t*)(pe + rel_rva + off + 8);
            for (int j = 0; j < cnt; j++) {
                if ((ents[j] >> 12) == 10) {
                    uint64_t* ptr = (uint64_t*)(g_image_base + pg + (ents[j] & 0xFFF));
                    *ptr += delta;
                }
            }
            off += blk;
        }
    }

    // Разрешение импорта (IAT)
    if (imp_rva) {
        uint8_t* desc = g_image_base + imp_rva;
        while (1) {
            uint32_t orig_thunk = *(uint32_t*)desc;
            uint32_t first_thunk = *(uint32_t*)(desc + 16);
            if (!orig_thunk && !first_thunk) break;
            uint32_t thunk_rva = orig_thunk ? orig_thunk : first_thunk;
            uint64_t* thunk = (uint64_t*)(g_image_base + thunk_rva);
            uint64_t* iat = (uint64_t*)(g_image_base + first_thunk);

            while (*thunk) {
                if (!(*thunk & (1ULL << 63))) {
                    char* name = (char*)(g_image_base + *thunk + 2);
                    void* sym = resolve_sym(name);
                    *iat = sym ? (uint64_t)sym : (uint64_t)dummy_stub;
                }
                thunk++;
                iat++;
            }
            desc += 20;
        }
    }

    memset(&g_teb, 0, sizeof(g_teb));
    memset(&g_peb, 0, sizeof(g_peb));
    g_teb.Self = &g_teb;
    g_teb.Peb = &g_peb;
    g_peb.ProcessHeap = (void*)0x50000;
    syscall(SYS_arch_prctl, ARCH_SET_GS, &g_teb);

    typedef MS_ABI void (*Entry_t)();
    Entry_t entry = (Entry_t)(g_image_base + entry_rva);

    printf("[*] Executing Windows Direct3D Application at %p...\\n", entry);
    if (setjmp(g_exit_env) == 0) {
        entry();
    }

    printf("[*] Direct3D Application finished successfully without Wine!\\n");
    return 0;
}
"""

with open("d3d_loader.c", "w") as f:
    f.write(loader_d3d_c)

# Компилируем Linux-лоадер
subprocess.run(["gcc", "-O3", "d3d_loader.c", "-o", "d3d_loader", "-lm"], check=True)

# Запускаем!
proc = subprocess.Popen(["./d3d_loader"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
out, _ = proc.communicate()
print(out)

[+] Windows Direct3D PE binary 'd3d_app.exe' compiled successfully.



In [15]:
%%bash
apt-get update -qq
apt-get install -y -qq gcc-mingw-w64-x86-64

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [16]:
%%writefile d3d_app.c
#include <windows.h>
#include <stdint.h>

struct IDirect3DDevice9;

struct IDirect3D9Vtbl {
    void* QueryInterface;
    void* AddRef;
    void* Release;
    void* RegisterSoftwareDevice;
    void* GetAdapterCount;
    void* GetAdapterIdentifier;
    void* GetAdapterModeCount;
    void* EnumAdapterModes;
    void* GetAdapterDisplayMode;
    void* CheckDeviceType;
    void* CheckDeviceFormat;
    void* CheckDeviceMultiSampleType;
    void* CheckDepthStencilMatch;
    void* CheckDeviceFormatConversion;
    void* GetDeviceCaps;
    void* GetAdapterMonitor;
    HRESULT (__attribute__((ms_abi)) *CreateDevice)(void* This, UINT Adapter, DWORD DeviceType, HWND hFocusWindow, DWORD BehaviorFlags, void* pParams, struct IDirect3DDevice9** ppDevice);
};

struct IDirect3D9 {
    struct IDirect3D9Vtbl* lpVtbl;
};

typedef struct {
    float x, y, z;
    uint32_t color; // 0xAARRGGBB
} Vertex;

struct IDirect3DDevice9Vtbl {
    void* vtbl[17];
    HRESULT (__attribute__((ms_abi)) *Present)(struct IDirect3DDevice9* This, const void* src, const void* dst, HWND wnd, const void* dirty);
    void* vtbl2[23];
    HRESULT (__attribute__((ms_abi)) *BeginScene)(struct IDirect3DDevice9* This);
    HRESULT (__attribute__((ms_abi)) *EndScene)(struct IDirect3DDevice9* This);
    HRESULT (__attribute__((ms_abi)) *Clear)(struct IDirect3DDevice9* This, DWORD Count, const void* pRects, DWORD Flags, uint32_t Color, float Z, DWORD Stencil);
    void* vtbl3[38];
    HRESULT (__attribute__((ms_abi)) *DrawPrimitiveUP)(struct IDirect3DDevice9* This, DWORD PrimitiveType, UINT PrimitiveCount, const void* pData, UINT Stride);
};

struct IDirect3DDevice9 {
    struct IDirect3DDevice9Vtbl* lpVtbl;
};

// Импорт фабрики Direct3D9
__declspec(dllimport) struct IDirect3D9* __attribute__((ms_abi)) Direct3DCreate9(UINT SDKVersion);

void mainCRTStartup() {
    struct IDirect3D9* d3d = Direct3DCreate9(32);
    struct IDirect3DDevice9* dev = NULL;

    d3d->lpVtbl->CreateDevice(d3d, 0, 1, NULL, 0, NULL, &dev);

    // 1. Очистка буфера (Темно-синий фон)
    dev->lpVtbl->Clear(dev, 0, NULL, 1, 0xFF101528, 1.0f, 0);

    // 2. Вершины цветного треугольника
    Vertex tri[3] = {
        { 40.0f,  4.0f, 0.0f, 0xFFFF2020 }, // Верхняя (Красная)
        { 70.0f, 26.0f, 0.0f, 0xFF20FF20 }, // Правая (Зеленая)
        { 10.0f, 26.0f, 0.0f, 0xFF2060FF }  // Левая (Синяя)
    };

    // 3. Отрисовка кадра
    dev->lpVtbl->BeginScene(dev);
    dev->lpVtbl->DrawPrimitiveUP(dev, 4, 1, tri, sizeof(Vertex));
    dev->lpVtbl->EndScene(dev);

    // 4. Презентация на экран
    dev->lpVtbl->Present(dev, NULL, NULL, NULL, NULL);

    ExitProcess(0);
}

Writing d3d_app.c


In [17]:
%%writefile d3d_loader.c
#define _GNU_SOURCE
#include <stdio.h>
#include <stdlib.h>
#include <stdint.h>
#include <string.h>
#include <sys/mman.h>
#include <unistd.h>
#include <setjmp.h>
#include <sys/syscall.h>
#include <asm/prctl.h>
#include <math.h>

#define MS_ABI __attribute__((ms_abi))
#define FB_W 80
#define FB_H 30

static uint32_t g_fb[FB_H][FB_W];
static jmp_buf g_exit_env;
static uint8_t* g_base = NULL;

typedef struct { uint8_t r[24]; void* Heap; } WIN_PEB;
typedef struct { void* r1[6]; void* Self; void* r2[4]; void* Tls; WIN_PEB* Peb; } WIN_TEB;
static WIN_TEB g_teb;
static WIN_PEB g_peb;

typedef struct { float x, y, z; uint32_t color; } D3DVertex;

// Программный растеризатор интерполяции RGB
static void draw_triangle(const D3DVertex* v0, const D3DVertex* v1, const D3DVertex* v2) {
    int min_x = (int)fmaxf(0, fminf(v0->x, fminf(v1->x, v2->x)));
    int max_x = (int)fminf(FB_W - 1, fmaxf(v0->x, fmaxf(v1->x, v2->x)));
    int min_y = (int)fmaxf(0, fminf(v0->y, fminf(v1->y, v2->y)));
    int max_y = (int)fminf(FB_H - 1, fmaxf(v0->y, fmaxf(v1->y, v2->y)));

    float denom = (v1->y - v2->y) * (v0->x - v2->x) + (v2->x - v1->x) * (v0->y - v2->y);
    if (fabsf(denom) < 1e-5f) return;

    for (int y = min_y; y <= max_y; y++) {
        for (int x = min_x; x <= max_x; x++) {
            float px = (float)x + 0.5f, py = (float)y + 0.5f;
            float w0 = ((v1->y - v2->y) * (px - v2->x) + (v2->x - v1->x) * (py - v2->y)) / denom;
            float w1 = ((v2->y - v0->y) * (px - v2->x) + (v0->x - v2->x) * (py - v2->y)) / denom;
            float w2 = 1.0f - w0 - w1;

            if (w0 >= 0 && w1 >= 0 && w2 >= 0) {
                uint8_t r = (uint8_t)(w0 * ((v0->color >> 16) & 0xFF) + w1 * ((v1->color >> 16) & 0xFF) + w2 * ((v2->color >> 16) & 0xFF));
                uint8_t g = (uint8_t)(w0 * ((v0->color >> 8) & 0xFF)  + w1 * ((v1->color >> 8) & 0xFF)  + w2 * ((v2->color >> 8) & 0xFF));
                uint8_t b = (uint8_t)(w0 * (v0->color & 0xFF)         + w1 * (v1->color & 0xFF)         + w2 * (v2->color & 0xFF));
                g_fb[y][x] = (0xFF << 24) | (r << 16) | (g << 8) | b;
            }
        }
    }
}

// COM Методы
MS_ABI int dev_Clear(void* t, uint32_t c, const void* p, uint32_t f, uint32_t col, float z, uint32_t s) {
    for (int y = 0; y < FB_H; y++)
        for (int x = 0; x < FB_W; x++) g_fb[y][x] = col;
    return 0;
}
MS_ABI int dev_BeginScene(void* t) { return 0; }
MS_ABI int dev_EndScene(void* t) { return 0; }
MS_ABI int dev_DrawPrimitiveUP(void* t, uint32_t pt, uint32_t count, const void* data, uint32_t str) {
    const D3DVertex* v = (const D3DVertex*)data;
    for (uint32_t i = 0; i < count; i++) draw_triangle(&v[i*3], &v[i*3+1], &v[i*3+2]);
    return 0;
}
MS_ABI int dev_Present(void* t, const void* s, const void* d, void* w, const void* r) {
    printf("\n=== [DirectX 9 Framebuffer Output in Linux Terminal] ===\n\n");
    for (int y = 0; y < FB_H; y += 2) {
        for (int x = 0; x < FB_W; x++) {
            uint32_t top = g_fb[y][x];
            uint32_t bot = (y + 1 < FB_H) ? g_fb[y + 1][x] : top;
            uint8_t tr = (top >> 16) & 0xFF, tg = (top >> 8) & 0xFF, tb = top & 0xFF;
            uint8_t br = (bot >> 16) & 0xFF, bg = (bot >> 8) & 0xFF, bb = bot & 0xFF;
            printf("\033[38;2;%d;%d;%dm\033[48;2;%d;%d;%dm▀", tr, tg, tb, br, bg, bb);
        }
        printf("\033[0m\n");
    }
    printf("\033[0m\n");
    fflush(stdout);
    return 0;
}

static void* g_dev_vtbl[100];
static struct { void* vtbl; } g_dev_inst = { g_dev_vtbl };

MS_ABI int d3d9_CreateDevice(void* t, uint32_t a, uint32_t dt, void* w, uint32_t f, void* p, void** ppDev) {
    g_dev_vtbl[17] = (void*)dev_Present;
    g_dev_vtbl[41] = (void*)dev_BeginScene;
    g_dev_vtbl[42] = (void*)dev_EndScene;
    g_dev_vtbl[43] = (void*)dev_Clear;
    g_dev_vtbl[82] = (void*)dev_DrawPrimitiveUP;
    *ppDev = &g_dev_inst;
    return 0;
}

static void* g_d3d_vtbl[30];
static struct { void* vtbl; } g_d3d_inst = { g_d3d_vtbl };

MS_ABI void* hook_Direct3DCreate9(uint32_t ver) {
    g_d3d_vtbl[16] = (void*)d3d9_CreateDevice;
    return &g_d3d_inst;
}
MS_ABI void hook_ExitProcess(uint32_t code) { longjmp(g_exit_env, 1); }
MS_ABI uint64_t dummy_stub(void) { return 0; }

int main() {
    FILE* f = fopen("d3d_app.exe", "rb");
    if (!f) { printf("[-] Failed to open d3d_app.exe\n"); return 1; }
    fseek(f, 0, SEEK_END);
    size_t sz = ftell(f);
    fseek(f, 0, SEEK_SET);
    uint8_t* pe = malloc(sz);
    fread(pe, 1, sz, f);
    fclose(f);

    uint32_t elfanew = *(uint32_t*)(pe + 0x3C);
    uint16_t nsec = *(uint16_t*)(pe + elfanew + 0x6);
    uint8_t* opt = pe + elfanew + 0x18;
    uint64_t pref = *(uint64_t*)(opt + 0x18);
    uint32_t entry_rva = *(uint32_t*)(opt + 0x10);
    uint32_t img_sz = *(uint32_t*)(opt + 0x38);
    uint32_t hdr_sz = *(uint32_t*)(opt + 0x3C);
    uint8_t* dirs = opt + 0x70;
    uint32_t imp_rva = *(uint32_t*)(dirs + 0x8);
    uint32_t rel_rva = *(uint32_t*)(dirs + 0x28);
    uint32_t rel_sz  = *(uint32_t*)(dirs + 0x2C);

    g_base = (uint8_t*)mmap((void*)pref, img_sz, PROT_READ | PROT_WRITE | PROT_EXEC, MAP_PRIVATE | MAP_ANONYMOUS, -1, 0);
    if (g_base == MAP_FAILED) g_base = (uint8_t*)mmap(NULL, img_sz, PROT_READ | PROT_WRITE | PROT_EXEC, MAP_PRIVATE | MAP_ANONYMOUS, -1, 0);

    memcpy(g_base, pe, hdr_sz);
    uint8_t* sec_hdr = opt + *(uint16_t*)(pe + elfanew + 0x14);
    for (int i = 0; i < nsec; i++) {
        uint8_t* s = sec_hdr + i * 40;
        uint32_t va = *(uint32_t*)(s + 12), vs = *(uint32_t*)(s + 8), rs = *(uint32_t*)(s + 16), rp = *(uint32_t*)(s + 20);
        if (rs > 0) memcpy(g_base + va, pe + rp, rs);
        if (vs > rs) memset(g_base + va + rs, 0, vs - rs);
    }

    // Релокации
    int64_t delta = (int64_t)g_base - (int64_t)pref;
    if (rel_rva && delta != 0) {
        uint32_t off = 0;
        while (off < rel_sz) {
            uint32_t pg = *(uint32_t*)(pe + rel_rva + off), blk = *(uint32_t*)(pe + rel_rva + off + 4);
            if (!blk) break;
            int cnt = (blk - 8) / 2;
            uint16_t* ents = (uint16_t*)(pe + rel_rva + off + 8);
            for (int j = 0; j < cnt; j++) {
                if ((ents[j] >> 12) == 10) {
                    uint64_t* ptr = (uint64_t*)(g_base + pg + (ents[j] & 0xFFF));
                    *ptr += delta;
                }
            }
            off += blk;
        }
    }

    // Патчинг IAT
    if (imp_rva) {
        uint8_t* desc = g_base + imp_rva;
        while (1) {
            uint32_t orig = *(uint32_t*)desc, ft = *(uint32_t*)(desc + 16);
            if (!orig && !ft) break;
            uint32_t th_rva = orig ? orig : ft;
            uint64_t* thunk = (uint64_t*)(g_base + th_rva);
            uint64_t* iat = (uint64_t*)(g_base + ft);
            while (*thunk) {
                if (!(*thunk & (1ULL << 63))) {
                    char* name = (char*)(g_base + *thunk + 2);
                    if (strcmp(name, "Direct3DCreate9") == 0) *iat = (uint64_t)hook_Direct3DCreate9;
                    else if (strcmp(name, "ExitProcess") == 0) *iat = (uint64_t)hook_ExitProcess;
                    else *iat = (uint64_t)dummy_stub;
                }
                thunk++; iat++;
            }
            desc += 20;
        }
    }

    memset(&g_teb, 0, sizeof(g_teb));
    memset(&g_peb, 0, sizeof(g_peb));
    g_teb.Self = &g_teb;
    g_teb.Peb = &g_peb;
    syscall(SYS_arch_prctl, ARCH_SET_GS, &g_teb);

    printf("[+] Direct3D Windows PE mapped at %p. Starting execution...\n", g_base);
    fflush(stdout);

    typedef MS_ABI void (*Entry_t)();
    Entry_t entry = (Entry_t)(g_base + entry_rva);

    if (setjmp(g_exit_env) == 0) {
        entry();
    }

    printf("[+] Direct3D execution completed successfully!\n");
    return 0;
}

Overwriting d3d_loader.c


In [18]:
%%bash
# 1. Создаем библиотеку импорта для DirectX 9
echo -e "LIBRARY d3d9.dll\nEXPORTS\nDirect3DCreate9\n" > d3d9.def
x86_64-w64-mingw32-dlltool -d d3d9.def -l libd3d9.a

# 2. Собираем Windows PE x64 бинарник
x86_64-w64-mingw32-gcc -nostdlib -O2 d3d_app.c -o d3d_app.exe -L. -ld3d9 -lkernel32

# 3. Собираем Linux-лоадер
gcc -O3 d3d_loader.c -o d3d_loader -lm

# 4. Запускаем напрямую
./d3d_loader

[+] Direct3D Windows PE mapped at 0x140000000. Starting execution...

=== [DirectX 9 Framebuffer Output in Linux Terminal] ===

▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀
▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀
▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀
▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀
▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀
▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀
▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀
▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀
▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀
▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀
▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀

d3d_loader.c: In function ‘main’:
d3d_loader.c:115:5: warning: ignoring return value of ‘fread’ declared with attribute ‘warn_unused_result’ [-Wunused-result]
  115 |     fread(pe, 1, sz, f);
      |     ^~~~~~~~~~~~~~~~~~~


In [22]:
%%writefile d3d_app.c
#include <windows.h>
#include <stdint.h>

struct IDirect3DDevice9;

struct IDirect3D9Vtbl {
    void* vtbl[16];
    HRESULT (__attribute__((ms_abi)) *CreateDevice)(void* This, UINT Adapter, DWORD DeviceType, HWND hFocusWindow, DWORD BehaviorFlags, void* pParams, struct IDirect3DDevice9** ppDevice);
};

struct IDirect3D9 {
    struct IDirect3D9Vtbl* lpVtbl;
};

typedef struct {
    float x, y, z;
    uint32_t color; // 0xAARRGGBB
} Vertex;

struct IDirect3DDevice9Vtbl {
    void* vtbl1[17];
    HRESULT (__attribute__((ms_abi)) *Present)(struct IDirect3DDevice9* This, const void* src, const void* dst, HWND wnd, const void* dirty);
    void* vtbl2[23];
    HRESULT (__attribute__((ms_abi)) *BeginScene)(struct IDirect3DDevice9* This);
    HRESULT (__attribute__((ms_abi)) *EndScene)(struct IDirect3DDevice9* This);
    HRESULT (__attribute__((ms_abi)) *Clear)(struct IDirect3DDevice9* This, DWORD Count, const void* pRects, DWORD Flags, uint32_t Color, float Z, DWORD Stencil);
    void* vtbl3[38];
    HRESULT (__attribute__((ms_abi)) *DrawPrimitiveUP)(struct IDirect3DDevice9* This, DWORD PrimitiveType, UINT PrimitiveCount, const void* pData, UINT Stride);
};

struct IDirect3DDevice9 {
    struct IDirect3DDevice9Vtbl* lpVtbl;
};

__declspec(dllimport) struct IDirect3D9* __attribute__((ms_abi)) Direct3DCreate9(UINT SDKVersion);

// Встроенная быстрая тригонометрия (без зависимости от libm/msvcrt)
#define PI 3.14159265f

static float my_sin(float x) {
    while (x > PI)  x -= 2.0f * PI;
    while (x < -PI) x += 2.0f * PI;
    // Ряд Тейлора 5-го порядка
    float x2 = x * x;
    return x * (1.0f - x2 * (1.0f / 6.0f - x2 * (1.0f / 120.0f)));
}

static float my_cos(float x) {
    return my_sin(x + (PI * 0.5f));
}

// 3D Математика
typedef struct { float x, y, z; } Vec3;
typedef struct { float m[4][4]; } Mat4;

static Mat4 mat_identity() {
    Mat4 r = {0};
    r.m[0][0] = r.m[1][1] = r.m[2][2] = r.m[3][3] = 1.0f;
    return r;
}

static Mat4 mat_mul(Mat4 a, Mat4 b) {
    Mat4 r = {0};
    for (int i = 0; i < 4; i++)
        for (int j = 0; j < 4; j++)
            for (int k = 0; k < 4; k++)
                r.m[i][j] += a.m[i][k] * b.m[k][j];
    return r;
}

static Mat4 mat_rotate_xy(float ax, float ay) {
    Mat4 rx = mat_identity(), ry = mat_identity();
    float sx = my_sin(ax), cx = my_cos(ax);
    float sy = my_sin(ay), cy = my_cos(ay);

    rx.m[1][1] = cx;  rx.m[1][2] = -sx;
    rx.m[2][1] = sx;  rx.m[2][2] = cx;

    ry.m[0][0] = cy;  ry.m[0][2] = sy;
    ry.m[2][0] = -sy; ry.m[2][2] = cy;

    return mat_mul(ry, rx);
}

static Mat4 mat_perspective(float fov, float aspect, float zn, float zf) {
    Mat4 r = {0};
    float t = my_sin(fov * 0.5f) / my_cos(fov * 0.5f); // tan(fov/2)
    r.m[0][0] = 1.0f / (aspect * t);
    r.m[1][1] = 1.0f / t;
    r.m[2][2] = zf / (zf - zn);
    r.m[2][3] = 1.0f;
    r.m[3][2] = -(zf * zn) / (zf - zn);
    return r;
}

static Vertex transform_vertex(Vec3 in_v, uint32_t col, Mat4 mvp, float scr_w, float scr_h) {
    float x = in_v.x * mvp.m[0][0] + in_v.y * mvp.m[1][0] + in_v.z * mvp.m[2][0] + mvp.m[3][0];
    float y = in_v.x * mvp.m[0][1] + in_v.y * mvp.m[1][1] + in_v.z * mvp.m[2][1] + mvp.m[3][1];
    float z = in_v.x * mvp.m[0][2] + in_v.y * mvp.m[1][2] + in_v.z * mvp.m[2][2] + mvp.m[3][2];
    float w = in_v.x * mvp.m[0][3] + in_v.y * mvp.m[1][3] + in_v.z * mvp.m[2][3] + mvp.m[3][3];

    Vertex out_v;
    if (w <= 0.001f) w = 0.001f;
    float inv_w = 1.0f / w;
    float ndc_x = x * inv_w;
    float ndc_y = y * inv_w;
    float ndc_z = z * inv_w;

    out_v.x = (ndc_x + 1.0f) * 0.5f * scr_w;
    out_v.y = (1.0f - ndc_y) * 0.5f * scr_h;
    out_v.z = ndc_z;
    out_v.color = col;
    return out_v;
}

static const Vec3 CUBE_VERTICES[8] = {
    {-1.0f, -1.0f, -1.0f}, { 1.0f, -1.0f, -1.0f},
    { 1.0f,  1.0f, -1.0f}, {-1.0f,  1.0f, -1.0f},
    {-1.0f, -1.0f,  1.0f}, { 1.0f, -1.0f,  1.0f},
    { 1.0f,  1.0f,  1.0f}, {-1.0f,  1.0f,  1.0f}
};

static const int CUBE_INDICES[36] = {
    0, 2, 1,  0, 3, 2,
    4, 5, 6,  4, 6, 7,
    0, 4, 7,  0, 7, 3,
    1, 2, 6,  1, 6, 5,
    3, 7, 6,  3, 6, 2,
    0, 1, 5,  0, 5, 4
};

static const uint32_t FACE_COLORS[6] = {
    0xFFFF2525, // Красная
    0xFF25FF25, // Зеленая
    0xFF2565FF, // Синяя
    0xFFFFFF25, // Желтая
    0xFFFF25FF, // Пурпурная
    0xFF25FFFF  // Голубая
};

void mainCRTStartup() {
    struct IDirect3D9* d3d = Direct3DCreate9(32);
    struct IDirect3DDevice9* dev = NULL;
    d3d->lpVtbl->CreateDevice(d3d, 0, 1, NULL, 0, NULL, &dev);

    float scr_w = 80.0f, scr_h = 30.0f;
    Mat4 proj = mat_perspective(1.047f, (scr_w / scr_h) * 2.0f, 0.1f, 100.0f);

    Mat4 view = mat_identity();
    view.m[3][2] = 3.2f;

    for (int frame = 0; frame < 30; frame++) {
        float angle_x = frame * 0.15f;
        float angle_y = frame * 0.22f;

        Mat4 model = mat_rotate_xy(angle_x, angle_y);
        Mat4 mv = mat_mul(model, view);
        Mat4 mvp = mat_mul(mv, proj);

        dev->lpVtbl->Clear(dev, 0, NULL, 1, 0xFF080C14, 1.0f, 0);

        Vertex render_triangles[36];
        for (int i = 0; i < 12; i++) {
            int face = i / 2;
            uint32_t col = FACE_COLORS[face];
            render_triangles[i * 3 + 0] = transform_vertex(CUBE_VERTICES[CUBE_INDICES[i * 3 + 0]], col, mvp, scr_w, scr_h);
            render_triangles[i * 3 + 1] = transform_vertex(CUBE_VERTICES[CUBE_INDICES[i * 3 + 1]], col, mvp, scr_w, scr_h);
            render_triangles[i * 3 + 2] = transform_vertex(CUBE_VERTICES[CUBE_INDICES[i * 3 + 2]], col, mvp, scr_w, scr_h);
        }

        dev->lpVtbl->BeginScene(dev);
        dev->lpVtbl->DrawPrimitiveUP(dev, 4, 12, render_triangles, sizeof(Vertex));
        dev->lpVtbl->EndScene(dev);

        dev->lpVtbl->Present(dev, NULL, NULL, NULL, NULL);

        Sleep(35);
    }

    ExitProcess(0);
}

Overwriting d3d_app.c


In [23]:
%%writefile d3d_loader.c
#define _GNU_SOURCE
#include <stdio.h>
#include <stdlib.h>
#include <stdint.h>
#include <string.h>
#include <sys/mman.h>
#include <unistd.h>
#include <setjmp.h>
#include <sys/syscall.h>
#include <asm/prctl.h>
#include <math.h>

#define MS_ABI __attribute__((ms_abi))
#define FB_W 80
#define FB_H 30

static uint32_t g_fb[FB_H][FB_W];
static float g_zbuffer[FB_H][FB_W];
static jmp_buf g_exit_env;
static uint8_t* g_base = NULL;
static int g_first_frame = 1;

typedef struct { uint8_t r[24]; void* Heap; } WIN_PEB;
typedef struct { void* r1[6]; void* Self; void* r2[4]; void* Tls; WIN_PEB* Peb; } WIN_TEB;
static WIN_TEB g_teb;
static WIN_PEB g_peb;

typedef struct { float x, y, z; uint32_t color; } D3DVertex;

static void draw_triangle_3d(const D3DVertex* v0, const D3DVertex* v1, const D3DVertex* v2) {
    int min_x = (int)fmaxf(0, fminf(v0->x, fminf(v1->x, v2->x)));
    int max_x = (int)fminf(FB_W - 1, fmaxf(v0->x, fmaxf(v1->x, v2->x)));
    int min_y = (int)fmaxf(0, fminf(v0->y, fminf(v1->y, v2->y)));
    int max_y = (int)fminf(FB_H - 1, fmaxf(v0->y, fmaxf(v1->y, v2->y)));

    float denom = (v1->y - v2->y) * (v0->x - v2->x) + (v2->x - v1->x) * (v0->y - v2->y);
    if (fabsf(denom) < 1e-5f) return;

    for (int y = min_y; y <= max_y; y++) {
        for (int x = min_x; x <= max_x; x++) {
            float px = (float)x + 0.5f, py = (float)y + 0.5f;
            float w0 = ((v1->y - v2->y) * (px - v2->x) + (v2->x - v1->x) * (py - v2->y)) / denom;
            float w1 = ((v2->y - v0->y) * (px - v2->x) + (v0->x - v2->x) * (py - v2->y)) / denom;
            float w2 = 1.0f - w0 - w1;

            if (w0 >= 0 && w1 >= 0 && w2 >= 0) {
                float z = w0 * v0->z + w1 * v1->z + w2 * v2->z;
                if (z > 0.0f && z < g_zbuffer[y][x]) {
                    g_zbuffer[y][x] = z;

                    uint8_t r = (uint8_t)(w0 * ((v0->color >> 16) & 0xFF) + w1 * ((v1->color >> 16) & 0xFF) + w2 * ((v2->color >> 16) & 0xFF));
                    uint8_t g = (uint8_t)(w0 * ((v0->color >> 8) & 0xFF)  + w1 * ((v1->color >> 8) & 0xFF)  + w2 * ((v2->color >> 8) & 0xFF));
                    uint8_t b = (uint8_t)(w0 * (v0->color & 0xFF)         + w1 * (v1->color & 0xFF)         + w2 * (v2->color & 0xFF));

                    g_fb[y][x] = (0xFF << 24) | (r << 16) | (g << 8) | b;
                }
            }
        }
    }
}

MS_ABI int dev_Clear(void* t, uint32_t c, const void* p, uint32_t f, uint32_t col, float z, uint32_t s) {
    for (int y = 0; y < FB_H; y++) {
        for (int x = 0; x < FB_W; x++) {
            g_fb[y][x] = col;
            g_zbuffer[y][x] = 1000.0f;
        }
    }
    return 0;
}
MS_ABI int dev_BeginScene(void* t) { return 0; }
MS_ABI int dev_EndScene(void* t) { return 0; }
MS_ABI int dev_DrawPrimitiveUP(void* t, uint32_t pt, uint32_t count, const void* data, uint32_t str) {
    const D3DVertex* v = (const D3DVertex*)data;
    for (uint32_t i = 0; i < count; i++) {
        draw_triangle_3d(&v[i*3], &v[i*3+1], &v[i*3+2]);
    }
    return 0;
}

MS_ABI int dev_Present(void* t, const void* s, const void* d, void* w, const void* r) {
    if (g_first_frame) {
        printf("\033[2J\033[H\033[?25l");
        g_first_frame = 0;
    } else {
        printf("\033[H");
    }

    printf("=== [Direct3D 9 Spinning 3D Cube via Native Linux Meta-Loader] ===\n");
    for (int y = 0; y < FB_H; y += 2) {
        for (int x = 0; x < FB_W; x++) {
            uint32_t top = g_fb[y][x];
            uint32_t bot = (y + 1 < FB_H) ? g_fb[y + 1][x] : top;
            uint8_t tr = (top >> 16) & 0xFF, tg = (top >> 8) & 0xFF, tb = top & 0xFF;
            uint8_t br = (bot >> 16) & 0xFF, bg = (bot >> 8) & 0xFF, bb = bot & 0xFF;
            printf("\033[38;2;%d;%d;%dm\033[48;2;%d;%d;%dm▀", tr, tg, tb, br, bg, bb);
        }
        printf("\033[0m\n");
    }
    printf("\033[0m");
    fflush(stdout);
    return 0;
}

static void* g_dev_vtbl[100];
static struct { void* vtbl; } g_dev_inst = { g_dev_vtbl };

MS_ABI int d3d9_CreateDevice(void* t, uint32_t a, uint32_t dt, void* w, uint32_t f, void* p, void** ppDev) {
    g_dev_vtbl[17] = (void*)dev_Present;
    g_dev_vtbl[41] = (void*)dev_BeginScene;
    g_dev_vtbl[42] = (void*)dev_EndScene;
    g_dev_vtbl[43] = (void*)dev_Clear;
    g_dev_vtbl[82] = (void*)dev_DrawPrimitiveUP;
    *ppDev = &g_dev_inst;
    return 0;
}

static void* g_d3d_vtbl[30];
static struct { void* vtbl; } g_d3d_inst = { g_d3d_vtbl };

MS_ABI void* hook_Direct3DCreate9(uint32_t ver) {
    g_d3d_vtbl[16] = (void*)d3d9_CreateDevice;
    return &g_d3d_inst;
}
MS_ABI void hook_ExitProcess(uint32_t code) {
    printf("\033[?25h\n");
    longjmp(g_exit_env, 1);
}
MS_ABI void hook_Sleep(uint32_t ms) {
    usleep(ms * 1000);
}
MS_ABI uint64_t dummy_stub(void) { return 0; }

int main() {
    FILE* f = fopen("d3d_app.exe", "rb");
    if (!f) { printf("[-] Failed to open d3d_app.exe\n"); return 1; }
    fseek(f, 0, SEEK_END);
    size_t sz = ftell(f);
    fseek(f, 0, SEEK_SET);
    uint8_t* pe = malloc(sz);
    if (!pe || fread(pe, 1, sz, f) != sz) { fclose(f); return 1; }
    fclose(f);

    uint32_t elfanew = *(uint32_t*)(pe + 0x3C);
    uint16_t nsec = *(uint16_t*)(pe + elfanew + 0x6);
    uint8_t* opt = pe + elfanew + 0x18;
    uint64_t pref = *(uint64_t*)(opt + 0x18);
    uint32_t entry_rva = *(uint32_t*)(opt + 0x10);
    uint32_t img_sz = *(uint32_t*)(opt + 0x38);
    uint32_t hdr_sz = *(uint32_t*)(opt + 0x3C);
    uint8_t* dirs = opt + 0x70;
    uint32_t imp_rva = *(uint32_t*)(dirs + 0x8);
    uint32_t rel_rva = *(uint32_t*)(dirs + 0x28);
    uint32_t rel_sz  = *(uint32_t*)(dirs + 0x2C);

    g_base = (uint8_t*)mmap((void*)pref, img_sz, PROT_READ | PROT_WRITE | PROT_EXEC, MAP_PRIVATE | MAP_ANONYMOUS, -1, 0);
    if (g_base == MAP_FAILED) g_base = (uint8_t*)mmap(NULL, img_sz, PROT_READ | PROT_WRITE | PROT_EXEC, MAP_PRIVATE | MAP_ANONYMOUS, -1, 0);

    memcpy(g_base, pe, hdr_sz);
    uint8_t* sec_hdr = opt + *(uint16_t*)(pe + elfanew + 0x14);
    for (int i = 0; i < nsec; i++) {
        uint8_t* s = sec_hdr + i * 40;
        uint32_t va = *(uint32_t*)(s + 12), vs = *(uint32_t*)(s + 8), rs = *(uint32_t*)(s + 16), rp = *(uint32_t*)(s + 20);
        if (rs > 0) memcpy(g_base + va, pe + rp, rs);
        if (vs > rs) memset(g_base + va + rs, 0, vs - rs);
    }

    int64_t delta = (int64_t)g_base - (int64_t)pref;
    if (rel_rva && delta != 0) {
        uint32_t off = 0;
        while (off < rel_sz) {
            uint32_t pg = *(uint32_t*)(pe + rel_rva + off), blk = *(uint32_t*)(pe + rel_rva + off + 4);
            if (!blk) break;
            int cnt = (blk - 8) / 2;
            uint16_t* ents = (uint16_t*)(pe + rel_rva + off + 8);
            for (int j = 0; j < cnt; j++) {
                if ((ents[j] >> 12) == 10) {
                    uint64_t* ptr = (uint64_t*)(g_base + pg + (ents[j] & 0xFFF));
                    *ptr += delta;
                }
            }
            off += blk;
        }
    }

    if (imp_rva) {
        uint8_t* desc = g_base + imp_rva;
        while (1) {
            uint32_t orig = *(uint32_t*)desc, ft = *(uint32_t*)(desc + 16);
            if (!orig && !ft) break;
            uint32_t th_rva = orig ? orig : ft;
            uint64_t* thunk = (uint64_t*)(g_base + th_rva);
            uint64_t* iat = (uint64_t*)(g_base + ft);
            while (*thunk) {
                if (!(*thunk & (1ULL << 63))) {
                    char* name = (char*)(g_base + *thunk + 2);
                    if (strcmp(name, "Direct3DCreate9") == 0) *iat = (uint64_t)hook_Direct3DCreate9;
                    else if (strcmp(name, "ExitProcess") == 0) *iat = (uint64_t)hook_ExitProcess;
                    else if (strcmp(name, "Sleep") == 0) *iat = (uint64_t)hook_Sleep;
                    else *iat = (uint64_t)dummy_stub;
                }
                thunk++; iat++;
            }
            desc += 20;
        }
    }

    memset(&g_teb, 0, sizeof(g_teb));
    memset(&g_peb, 0, sizeof(g_peb));
    g_teb.Self = &g_teb;
    g_teb.Peb = &g_peb;
    syscall(SYS_arch_prctl, ARCH_SET_GS, &g_teb);

    typedef MS_ABI void (*Entry_t)();
    Entry_t entry = (Entry_t)(g_base + entry_rva);

    if (setjmp(g_exit_env) == 0) {
        entry();
    }

    printf("[+] Direct3D 3D-Cube Animation completed successfully!\n");
    return 0;
}

Overwriting d3d_loader.c


In [24]:
%%bash
# 1. Библиотека импорта d3d9.dll
echo -e "LIBRARY d3d9.dll\nEXPORTS\nDirect3DCreate9\n" > d3d9.def
x86_64-w64-mingw32-dlltool -d d3d9.def -l libd3d9.a

# 2. Компилируем чистый Windows PE x64 бинарник
x86_64-w64-mingw32-gcc -nostdlib -O2 d3d_app.c -o d3d_app.exe -L. -ld3d9 -lkernel32

# 3. Компилируем Linux PE-лоадер
gcc -O3 d3d_loader.c -o d3d_loader -lm

# 4. Запускаем анимацию
./d3d_loader

=== [Direct3D 9 Spinning 3D Cube via Native Linux Meta-Loader] ===
▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀
▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀
▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀
▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀
▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀
▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀
▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀
▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀
▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀
▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀
▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀
▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀▀

In [25]:
%%bash
# Скачиваем официальный пакет утилит Microsoft прямо с live.sysinternals.com
curl -s -L -o Strings.zip https://download.sysinternals.com/files/Strings.zip
unzip -q -o Strings.zip strings64.exe

# Проверяем, что это оригинальный PE32+ бинарник Microsoft Windows
file strings64.exe

strings64.exe: PE32+ executable (console) x86-64, for MS Windows


In [26]:
%%bash
# Смотрим секцию импортов PE через objdump / readelf
objdump -p strings64.exe | grep "DLL Name" -A 10

	DLL Name: VERSION.dll
	vma:  Hint/Ord Member-Name Bound-To
	6c8d2	   16  VerQueryValueW
	6c8c0	   15  VerQueryValueA
	6c8aa	    8  GetFileVersionInfoW
	6c894	    0  GetFileVersionInfoA
	6c87a	    7  GetFileVersionInfoSizeW
	6c860	    4  GetFileVersionInfoSizeA

 0006c3a0	0006c4a0 00000000 00000000 0006ca88 00054088

	DLL Name: KERNEL32.dll
	vma:  Hint/Ord Member-Name Bound-To
	6c99c	  806  GetVersionExA
	6c9ac	  969  LoadLibraryExA
	6c9be	  537  GetCurrentDirectoryA
	6c9d6	  198  CreateFileA
	6c9e4	  382  FindClose
	6c9f0	  386  FindFirstFileA
	6ca02	  403  FindNextFileA
	6ca12	  608  GetFullPathNameA
	6ca26	 1145  ReadFile
--
	DLL Name: USER32.dll
	vma:  Hint/Ord Member-Name Bound-To
	6cb1a	  597  LoadCursorA
	6cb0c	  522  InflateRect
	6caf8	  452  GetSysColorBrush
	6caec	  808  SetCursor
	6cada	  895  SetWindowTextA
	6cacc	  332  GetDlgItem
	6cac0	  242  EndDialog
	6caa6	  182  DialogBoxIndirectParamA
	6ca96	  787  SendMessageA
--
	DLL Name: GDI32.dll
	vma:  Hint/Ord Member-Name Bou

In [29]:
%%writefile ms_loader.c
#define _GNU_SOURCE
#include <stdio.h>
#include <stdlib.h>
#include <stdint.h>
#include <string.h>
#include <sys/mman.h>
#include <unistd.h>
#include <setjmp.h>
#include <sys/syscall.h>
#include <asm/prctl.h>
#include <fcntl.h>
#include <wchar.h>

#define MS_ABI __attribute__((ms_abi))

static jmp_buf g_exit_env;
static int g_exit_code = 0;
static uint8_t* g_base = NULL;

typedef struct { uint8_t r[24]; void* Heap; } WIN_PEB;
typedef struct { void* r1[6]; void* Self; void* r2[4]; void* Tls; WIN_PEB* Peb; } WIN_TEB;
static WIN_TEB g_teb;
static WIN_PEB g_peb;

typedef struct {
    uint32_t cb;
    void* lpReserved;
    void* lpDesktop;
    void* lpTitle;
    uint32_t dwX; uint32_t dwY; uint32_t dwXSize; uint32_t dwYSize;
    uint32_t dwXCountChars; uint32_t dwYCountChars; uint32_t dwFillAttribute;
    uint32_t dwFlags;
    uint16_t wShowWindow; uint16_t cbReserved2;
    void* lpReserved2;
    void* hStdInput; void* hStdOutput; void* hStdError;
} STARTUPINFOW;

// --- Системные Win32 API функции ---

MS_ABI void* hook_GetModuleHandleW(const wchar_t* n) { return g_base; }
MS_ABI void* hook_GetModuleHandleA(const char* n) { return g_base; }

MS_ABI wchar_t* hook_GetCommandLineW(void) {
    static wchar_t cmd[] = L"strings64.exe -accepteula -nobanner strings64.exe";
    return cmd;
}

MS_ABI char* hook_GetCommandLineA(void) {
    static char cmd[] = "strings64.exe -accepteula -nobanner strings64.exe";
    return cmd;
}

MS_ABI void* hook_GetStdHandle(uint32_t n) {
    if (n == (uint32_t)-11) return (void*)1; // STDOUT
    if (n == (uint32_t)-12) return (void*)2; // STDERR
    return (void*)0;                         // STDIN
}

MS_ABI void* hook_GetProcessHeap(void) { return (void*)0x50000; }
MS_ABI void* hook_HeapAlloc(void* h, uint32_t f, size_t s) { return calloc(1, s); }
MS_ABI int hook_HeapFree(void* h, uint32_t f, void* p) { free(p); return 1; }
MS_ABI void* hook_HeapReAlloc(void* h, uint32_t f, void* p, size_t s) { return realloc(p, s); }

MS_ABI int hook_WriteFile(void* hFile, const void* buf, uint32_t sz, uint32_t* wr, void* ov) {
    int fd = (int)(uintptr_t)hFile;
    if (fd != 1 && fd != 2) fd = 1;
    ssize_t res = write(fd, buf, sz);
    if (wr && res > 0) *wr = (uint32_t)res;
    return 1;
}

MS_ABI int hook_WriteConsoleW(void* h, const wchar_t* buf, uint32_t len, uint32_t* wr, void* r) {
    char mb[8192];
    size_t out_len = wcstombs(mb, buf, sizeof(mb) - 1);
    if (out_len != (size_t)-1) {
        ssize_t w = write(STDOUT_FILENO, mb, out_len);
        (void)w;
    }
    if (wr) *wr = len;
    return 1;
}

MS_ABI int hook_WriteConsoleA(void* h, const char* buf, uint32_t len, uint32_t* wr, void* r) {
    ssize_t res = write(STDOUT_FILENO, buf, len);
    if (wr && res > 0) *wr = (uint32_t)res;
    return 1;
}

MS_ABI void* hook_CreateFileW(const wchar_t* filename, uint32_t access, uint32_t share, void* sec, uint32_t disp, uint32_t flags, void* tmpl) {
    char name[512];
    wcstombs(name, filename, sizeof(name) - 1);
    int fd = open(name, O_RDONLY);
    if (fd < 0) return (void*)(intptr_t)-1;
    return (void*)(uintptr_t)(fd + 100);
}

MS_ABI int hook_ReadFile(void* hFile, void* buf, uint32_t count, uint32_t* read_bytes, void* ov) {
    int fd = (int)(uintptr_t)hFile - 100;
    if (fd < 0) return 0;
    ssize_t res = read(fd, buf, count);
    if (res >= 0) {
        if (read_bytes) *read_bytes = (uint32_t)res;
        return 1;
    }
    return 0;
}

MS_ABI int hook_CloseHandle(void* h) {
    int fd = (int)(uintptr_t)h - 100;
    if (fd >= 0) close(fd);
    return 1;
}

MS_ABI uint32_t hook_GetFileSize(void* hFile, uint32_t* high) {
    int fd = (int)(uintptr_t)hFile - 100;
    if (fd < 0) return 0;
    off_t cur = lseek(fd, 0, SEEK_CUR);
    off_t sz = lseek(fd, 0, SEEK_END);
    lseek(fd, cur, SEEK_SET);
    if (high) *high = (uint32_t)(sz >> 32);
    return (uint32_t)sz;
}

MS_ABI uint32_t hook_SetFilePointer(void* hFile, int32_t dist, int32_t* high, uint32_t method) {
    int fd = (int)(uintptr_t)hFile - 100;
    if (fd < 0) return (uint32_t)-1;
    int whence = (method == 0) ? SEEK_SET : ((method == 1) ? SEEK_CUR : SEEK_END);
    off_t res = lseek(fd, dist, whence);
    return (uint32_t)res;
}

MS_ABI void hook_GetStartupInfoW(STARTUPINFOW* si) {
    memset(si, 0, sizeof(STARTUPINFOW));
    si->cb = sizeof(STARTUPINFOW);
    si->hStdInput = (void*)0;
    si->hStdOutput = (void*)1;
    si->hStdError = (void*)2;
}

// Перехват реестра (Sysinternals EULA bypass)
MS_ABI int32_t hook_RegOpenKeyExW(void* k, const wchar_t* sub, uint32_t opt, uint32_t sam, void** res) {
    *res = (void*)0x90000;
    return 0; // ERROR_SUCCESS
}

MS_ABI int32_t hook_RegQueryValueExW(void* k, const wchar_t* val, uint32_t* r, uint32_t* type, uint8_t* data, uint32_t* cb) {
    if (type) *type = 4; // REG_DWORD
    if (data && cb && *cb >= 4) {
        *(uint32_t*)data = 1; // EulaAccepted = 1
        *cb = 4;
    }
    return 0; // ERROR_SUCCESS
}

MS_ABI int32_t hook_RegSetValueExW(void* k, const wchar_t* v, uint32_t r, uint32_t t, const uint8_t* d, uint32_t cb) { return 0; }
MS_ABI int32_t hook_RegCloseKey(void* k) { return 0; }

MS_ABI void hook_ExitProcess(uint32_t code) {
    g_exit_code = (int)code;
    longjmp(g_exit_env, 1);
}

MS_ABI uint32_t hook_GetFileType(void* h) { return 0x0002; } // FILE_TYPE_CHAR
MS_ABI int hook_GetConsoleMode(void* h, uint32_t* m) { if (m) *m = 7; return 1; }
MS_ABI int hook_SetConsoleMode(void* h, uint32_t m) { return 1; }
MS_ABI uint32_t hook_GetLastError(void) { return 0; }
MS_ABI void hook_SetLastError(uint32_t e) {}
MS_ABI void* hook_SetUnhandledExceptionFilter(void* f) { return NULL; }
MS_ABI uint32_t hook_GetCurrentProcessId(void) { return getpid(); }
MS_ABI uint32_t hook_GetCurrentThreadId(void) { return getpid(); }
MS_ABI void hook_GetSystemTimeAsFileTime(void* p) { memset(p, 0, 8); }
MS_ABI int hook_QueryPerformanceCounter(uint64_t* p) { if (p) *p = 1000; return 1; }
MS_ABI int hook_QueryPerformanceFrequency(uint64_t* p) { if (p) *p = 1000000; return 1; }
MS_ABI void* hook_VirtualAlloc(void* a, size_t sz, uint32_t t, uint32_t p) {
    return mmap(NULL, sz, PROT_READ | PROT_WRITE, MAP_PRIVATE | MAP_ANONYMOUS, -1, 0);
}
MS_ABI int hook_VirtualFree(void* a, size_t sz, uint32_t t) { munmap(a, sz); return 1; }
MS_ABI void* hook_GetEnvironmentStringsW(void) {
    static wchar_t env[] = L"PATH=/bin\0ALLUSERSPROFILE=C:\\ProgramData\0\0";
    return env;
}
MS_ABI int hook_FreeEnvironmentStringsW(void* p) { return 1; }

MS_ABI void* hook_GetProcAddress(void* h, const char* name);
MS_ABI void* hook_LoadLibraryA(const char* n) { return (void*)0x20000; }
MS_ABI void* hook_LoadLibraryW(const wchar_t* n) { return (void*)0x20000; }

typedef struct { const char* name; void* func; } Sym;
static Sym g_symbols[] = {
    {"GetModuleHandleW", hook_GetModuleHandleW},
    {"GetModuleHandleA", hook_GetModuleHandleA},
    {"GetCommandLineW", hook_GetCommandLineW},
    {"GetCommandLineA", hook_GetCommandLineA},
    {"GetStdHandle", hook_GetStdHandle},
    {"GetProcessHeap", hook_GetProcessHeap},
    {"HeapAlloc", hook_HeapAlloc},
    {"HeapFree", hook_HeapFree},
    {"HeapReAlloc", hook_HeapReAlloc},
    {"WriteFile", hook_WriteFile},
    {"ReadFile", hook_ReadFile},
    {"CreateFileW", hook_CreateFileW},
    {"CloseHandle", hook_CloseHandle},
    {"GetFileSize", hook_GetFileSize},
    {"SetFilePointer", hook_SetFilePointer},
    {"GetStartupInfoW", hook_GetStartupInfoW},
    {"RegOpenKeyExW", hook_RegOpenKeyExW},
    {"RegQueryValueExW", hook_RegQueryValueExW},
    {"RegSetValueExW", hook_RegSetValueExW},
    {"RegCloseKey", hook_RegCloseKey},
    {"WriteConsoleW", hook_WriteConsoleW},
    {"WriteConsoleA", hook_WriteConsoleA},
    {"ExitProcess", hook_ExitProcess},
    {"GetFileType", hook_GetFileType},
    {"GetConsoleMode", hook_GetConsoleMode},
    {"SetConsoleMode", hook_SetConsoleMode},
    {"GetLastError", hook_GetLastError},
    {"SetLastError", hook_SetLastError},
    {"SetUnhandledExceptionFilter", hook_SetUnhandledExceptionFilter},
    {"GetCurrentProcessId", hook_GetCurrentProcessId},
    {"GetCurrentThreadId", hook_GetCurrentThreadId},
    {"GetSystemTimeAsFileTime", hook_GetSystemTimeAsFileTime},
    {"QueryPerformanceCounter", hook_QueryPerformanceCounter},
    {"QueryPerformanceFrequency", hook_QueryPerformanceFrequency},
    {"VirtualAlloc", hook_VirtualAlloc},
    {"VirtualFree", hook_VirtualFree},
    {"GetEnvironmentStringsW", hook_GetEnvironmentStringsW},
    {"FreeEnvironmentStringsW", hook_FreeEnvironmentStringsW},
    {"GetProcAddress", hook_GetProcAddress},
    {"LoadLibraryA", hook_LoadLibraryA},
    {"LoadLibraryW", hook_LoadLibraryW},
    {NULL, NULL}
};

void* resolve_sym(const char* name) {
    for (int i = 0; g_symbols[i].name; i++) {
        if (strcmp(g_symbols[i].name, name) == 0) return g_symbols[i].func;
    }
    return NULL;
}

MS_ABI void* hook_GetProcAddress(void* h, const char* name) { return resolve_sym(name); }
MS_ABI uint64_t dummy_stub(void) { return 0; }

int main() {
    FILE* f = fopen("strings64.exe", "rb");
    if (!f) { printf("[-] strings64.exe not found!\n"); return 1; }
    fseek(f, 0, SEEK_END);
    size_t sz = ftell(f);
    fseek(f, 0, SEEK_SET);
    uint8_t* pe = malloc(sz);
    if (!pe || fread(pe, 1, sz, f) != sz) return 1;
    fclose(f);

    uint32_t elfanew = *(uint32_t*)(pe + 0x3C);
    uint16_t nsec = *(uint16_t*)(pe + elfanew + 0x6);
    uint8_t* opt = pe + elfanew + 0x18;
    uint64_t pref = *(uint64_t*)(opt + 0x18);
    uint32_t entry_rva = *(uint32_t*)(opt + 0x10);
    uint32_t img_sz = *(uint32_t*)(opt + 0x38);
    uint32_t hdr_sz = *(uint32_t*)(opt + 0x3C);
    uint8_t* dirs = opt + 0x70;
    uint32_t imp_rva = *(uint32_t*)(dirs + 0x8);
    uint32_t rel_rva = *(uint32_t*)(dirs + 0x28);
    uint32_t rel_sz  = *(uint32_t*)(dirs + 0x2C);

    g_base = (uint8_t*)mmap((void*)pref, img_sz, PROT_READ | PROT_WRITE | PROT_EXEC, MAP_PRIVATE | MAP_ANONYMOUS, -1, 0);
    if (g_base == MAP_FAILED) g_base = (uint8_t*)mmap(NULL, img_sz, PROT_READ | PROT_WRITE | PROT_EXEC, MAP_PRIVATE | MAP_ANONYMOUS, -1, 0);

    memcpy(g_base, pe, hdr_sz);
    uint8_t* sec_hdr = opt + *(uint16_t*)(pe + elfanew + 0x14);
    for (int i = 0; i < nsec; i++) {
        uint8_t* s = sec_hdr + i * 40;
        uint32_t va = *(uint32_t*)(s + 12), vs = *(uint32_t*)(s + 8), rs = *(uint32_t*)(s + 16), rp = *(uint32_t*)(s + 20);
        if (rs > 0) memcpy(g_base + va, pe + rp, rs);
        if (vs > rs) memset(g_base + va + rs, 0, vs - rs);
    }

    // Релокации
    int64_t delta = (int64_t)g_base - (int64_t)pref;
    if (rel_rva && delta != 0) {
        uint32_t off = 0;
        while (off < rel_sz) {
            uint32_t pg = *(uint32_t*)(pe + rel_rva + off), blk = *(uint32_t*)(pe + rel_rva + off + 4);
            if (!blk) break;
            int cnt = (blk - 8) / 2;
            uint16_t* ents = (uint16_t*)(pe + rel_rva + off + 8);
            for (int j = 0; j < cnt; j++) {
                if ((ents[j] >> 12) == 10) {
                    uint64_t* ptr = (uint64_t*)(g_base + pg + (ents[j] & 0xFFF));
                    *ptr += delta;
                }
            }
            off += blk;
        }
    }

    // Разрешение импортов IAT
    if (imp_rva) {
        uint8_t* desc = g_base + imp_rva;
        while (1) {
            uint32_t orig = *(uint32_t*)desc, ft = *(uint32_t*)(desc + 16);
            if (!orig && !ft) break;
            uint32_t th_rva = orig ? orig : ft;
            uint64_t* thunk = (uint64_t*)(g_base + th_rva);
            uint64_t* iat = (uint64_t*)(g_base + ft);
            while (*thunk) {
                if (!(*thunk & (1ULL << 63))) {
                    char* name = (char*)(g_base + *thunk + 2);
                    void* sym = resolve_sym(name);
                    *iat = sym ? (uint64_t)sym : (uint64_t)dummy_stub;
                }
                thunk++; iat++;
            }
            desc += 20;
        }
    }

    memset(&g_teb, 0, sizeof(g_teb));
    memset(&g_peb, 0, sizeof(g_peb));
    g_teb.Self = &g_teb;
    g_teb.Peb = &g_peb;
    g_peb.Heap = (void*)0x50000;
    syscall(SYS_arch_prctl, ARCH_SET_GS, &g_teb);

    printf("[+] Executing Microsoft Windows strings64.exe at %p...\n\n", g_base);
    fflush(stdout);

    typedef MS_ABI void (*Entry_t)();
    Entry_t entry = (Entry_t)(g_base + entry_rva);

    if (setjmp(g_exit_env) == 0) {
        entry();
    }

    printf("\n[+] Official Microsoft Binary finished with ExitCode: %d\n", g_exit_code);
    return 0;
}

Overwriting ms_loader.c


In [30]:
%%bash
gcc -O3 ms_loader.c -o ms_loader
./ms_loader | head -n 35

[+] Executing Microsoft Windows strings64.exe at 0x140000000...



In [33]:
%%bash
# Скачиваем официальный Windows x64 релиз ripgrep
curl -sL -H "User-Agent: Mozilla/5.0" -o rg.zip https://github.com/BurntSushi/ripgrep/releases/download/14.1.0/ripgrep-14.1.0-x86_64-pc-windows-msvc.zip
unzip -q -o rg.zip
cp ripgrep-14.1.0-x86_64-pc-windows-msvc/rg.exe ./target_app.exe

# Проверяем, что это официальный Windows PE x64
file target_app.exe

target_app.exe: PE32+ executable (console) x86-64, for MS Windows


In [49]:
%%writefile rg_loader.c
#define _GNU_SOURCE
#include <stdio.h>
#include <stdlib.h>
#include <stdint.h>
#include <string.h>
#include <sys/mman.h>
#include <unistd.h>
#include <setjmp.h>
#include <sys/syscall.h>
#include <asm/prctl.h>
#include <signal.h>
#include <ucontext.h>
#include <time.h>
#include <wchar.h>

#define MS_ABI __attribute__((ms_abi, noinline))

static jmp_buf g_exit_env;
static int g_exit_code = 0;
static uint8_t* g_base = NULL;

void crash_handler(int sig, siginfo_t *info, void *ucontext) {
    ucontext_t *uc = (ucontext_t *)ucontext;
    uint64_t rip = uc->uc_mcontext.gregs[REG_RIP];
    uint64_t rsp = uc->uc_mcontext.gregs[REG_RSP];
    uint64_t rax = uc->uc_mcontext.gregs[REG_RAX];
    uint64_t rcx = uc->uc_mcontext.gregs[REG_RCX];
    uint64_t rdx = uc->uc_mcontext.gregs[REG_RDX];
    printf("\n[FATAL] Signal %d (%s) at %p\n", sig, strsignal(sig), info->si_addr);
    printf("        RIP: 0x%lx (PE Base: %p), RSP: 0x%lx\n", rip, g_base, rsp);
    printf("        RAX: 0x%lx, RCX: 0x%lx, RDX: 0x%lx\n", rax, rcx, rdx);
    fflush(stdout);
    _exit(128 + sig);
}

typedef struct {
    uint16_t Length;
    uint16_t MaximumLength;
    wchar_t* Buffer;
} UNICODE_STRING;

typedef struct {
    uint8_t Reserved1[16];
    void* Reserved2[5];
    UNICODE_STRING CurrentDirectoryPath;
    void* CurrentDirectoryHandle;
    UNICODE_STRING DllPath;
    UNICODE_STRING ImagePathName;
    UNICODE_STRING CommandLine;
} RTL_USER_PROCESS_PARAMETERS;

typedef struct {
    uint8_t Reserved1[2];
    uint8_t BeingDebugged;
    uint8_t Reserved2[21];
    void* Reserved3[4];
    RTL_USER_PROCESS_PARAMETERS* ProcessParameters;
    void* Reserved4[3];
    void* ProcessHeap;
    void* FastPebLock;
} WIN_PEB;

typedef struct {
    void* ExceptionList;
    void* StackBase;
    void* StackLimit;
    void* SubSystemTib;
    void* FiberData;
    void* ArbitraryUserPointer;
    void* Self;
    void* EnvironmentPointer;
    uint64_t ProcessId;
    uint64_t ThreadId;
    void* RpcHandle;
    void** TlsArray;
    WIN_PEB* ProcessEnvironmentBlock;
} __attribute__((packed)) WIN_TEB;

static WIN_TEB g_teb;
static WIN_PEB g_peb;
static RTL_USER_PROCESS_PARAMETERS g_proc_params;
static void* g_tls_slots[256];
static void* g_tls_ptrs[16];
static uint64_t g_fast_peb_lock = 0;

static wchar_t g_cmdline[] = L"rg.exe --version";
static wchar_t g_imagepath[] = L"C:\\target_app.exe";
static char g_cmdline_a[] = "rg.exe --version";

static uint8_t g_heap_buf[32 * 1024 * 1024];
static size_t g_heap_offset = 0;

// --- Системные Win32 API функции ---

MS_ABI void* hook_GetModuleHandleA(const char* n) { return g_base; }
MS_ABI void* hook_GetModuleHandleW(const wchar_t* n) { return g_base; }
MS_ABI int hook_GetModuleHandleExW(uint32_t flags, const wchar_t* n, void** mod) {
    if (mod) *mod = g_base;
    return 1;
}

MS_ABI wchar_t* hook_GetCommandLineW(void) { return g_cmdline; }
MS_ABI char* hook_GetCommandLineA(void) { return g_cmdline_a; }

MS_ABI void* hook_GetStdHandle(uint32_t n) {
    if (n == (uint32_t)-11) return (void*)1; // STDOUT
    if (n == (uint32_t)-12) return (void*)2; // STDERR
    return (void*)0;
}

MS_ABI void* hook_GetProcessHeap(void) { return (void*)g_heap_buf; }

MS_ABI void* hook_HeapAlloc(void* h, uint32_t f, size_t s) {
    if (s == 0) s = 1;
    s = (s + 15) & ~15;
    if (g_heap_offset + s < sizeof(g_heap_buf)) {
        void* ptr = &g_heap_buf[g_heap_offset];
        g_heap_offset += s;
        memset(ptr, 0, s);
        return ptr;
    }
    return NULL;
}

MS_ABI int hook_HeapFree(void* h, uint32_t f, void* p) { return 1; }

MS_ABI void* hook_HeapReAlloc(void* h, uint32_t f, void* p, size_t s) {
    void* n = hook_HeapAlloc(h, f, s);
    if (n && p) memcpy(n, p, s > 128 ? 128 : s);
    return n;
}

MS_ABI int hook_WriteFile(void* hFile, const void* buf, uint32_t sz, uint32_t* wr, void* ov) {
    int fd = (hFile == (void*)2) ? 2 : 1;
    ssize_t res = syscall(SYS_write, fd, buf, (size_t)sz);
    if (wr && res > 0) *wr = (uint32_t)res;
    return 1;
}

MS_ABI int hook_WriteConsoleA(void* h, const char* buf, uint32_t len, uint32_t* wr, void* r) {
    ssize_t res = syscall(SYS_write, 1, buf, (size_t)len);
    if (wr && res > 0) *wr = (uint32_t)res;
    return 1;
}

MS_ABI int hook_WriteConsoleW(void* h, const wchar_t* buf, uint32_t len, uint32_t* wr, void* r) {
    char mb[8192];
    size_t out_len = 0;
    for (uint32_t i = 0; i < len && out_len < sizeof(mb) - 1; i++) {
        if (buf[i] < 128) mb[out_len++] = (char)buf[i];
        else mb[out_len++] = '?';
    }
    syscall(SYS_write, 1, mb, out_len);
    if (wr) *wr = len;
    return 1;
}

MS_ABI void hook_ExitProcess(uint32_t code) {
    g_exit_code = (int)code;
    longjmp(g_exit_env, 1);
}

MS_ABI uint32_t hook_GetFileType(void* h) { return 0x0002; }
MS_ABI int hook_GetConsoleMode(void* h, uint32_t* m) { if (m) *m = 7; return 1; }
MS_ABI int hook_SetConsoleMode(void* h, uint32_t m) { return 1; }
MS_ABI uint32_t hook_GetLastError(void) { return 0; }
MS_ABI void hook_SetLastError(uint32_t e) {}
MS_ABI void* hook_SetUnhandledExceptionFilter(void* f) { return NULL; }
MS_ABI void* hook_AddVectoredExceptionHandler(uint32_t first, void* handler) { return (void*)0x30000; }
MS_ABI uint32_t hook_GetCurrentProcessId(void) { return 1000; }
MS_ABI uint32_t hook_GetCurrentThreadId(void) { return 1000; }

MS_ABI void hook_GetSystemTimeAsFileTime(uint64_t* p) {
    if (p) *p = 133500000000000000ULL;
}
MS_ABI void hook_GetSystemTimePreciseAsFileTime(uint64_t* p) {
    if (p) *p = 133500000000000000ULL;
}

MS_ABI int hook_QueryPerformanceCounter(uint64_t* p) { if (p) *p = 10000000; return 1; }
MS_ABI int hook_QueryPerformanceFrequency(uint64_t* p) { if (p) *p = 10000000; return 1; }

MS_ABI void hook_GetSystemInfo(void* lpSysInfo) {
    uint32_t* p = (uint32_t*)lpSysInfo;
    p[0] = 9;     // AMD64
    p[1] = 4096;
    p[6] = 4;
}

MS_ABI void* hook_VirtualAlloc(void* a, size_t sz, uint32_t t, uint32_t p) {
    return hook_HeapAlloc(NULL, 0, sz);
}
MS_ABI int hook_VirtualFree(void* a, size_t sz, uint32_t t) { return 1; }

static uint32_t g_tls_idx_counter = 0;
MS_ABI uint32_t hook_TlsAlloc(void) { return g_tls_idx_counter++; }
MS_ABI void* hook_TlsGetValue(uint32_t idx) {
    if (idx < 256) return g_tls_slots[idx];
    return NULL;
}
MS_ABI int hook_TlsSetValue(uint32_t idx, void* val) {
    if (idx < 256) {
        g_tls_slots[idx] = val;
        return 1;
    }
    return 0;
}
MS_ABI int hook_TlsFree(uint32_t idx) { return 1; }

MS_ABI void hook_AcquireSRWLockExclusive(void* p) {}
MS_ABI void hook_ReleaseSRWLockExclusive(void* p) {}
MS_ABI void hook_AcquireSRWLockShared(void* p) {}
MS_ABI void hook_ReleaseSRWLockShared(void* p) {}
MS_ABI void hook_InitializeSRWLock(void* p) {}
MS_ABI void hook_InitializeCriticalSection(void* p) {}
MS_ABI int hook_InitializeCriticalSectionAndSpinCount(void* p, uint32_t sc) { return 1; }
MS_ABI void hook_EnterCriticalSection(void* p) {}
MS_ABI void hook_LeaveCriticalSection(void* p) {}
MS_ABI void hook_DeleteCriticalSection(void* p) {}
MS_ABI int hook_SleepConditionVariableSRW(void* cv, void* srw, uint32_t ms, uint32_t flags) { return 1; }
MS_ABI void hook_WakeAllConditionVariable(void* cv) {}
MS_ABI void hook_WakeConditionVariable(void* cv) {}

MS_ABI void* hook_RtlCaptureContext(void* c) { return NULL; }
MS_ABI void* hook_RtlLookupFunctionEntry(uint64_t rip, uint64_t* base, void* hist) { return NULL; }
MS_ABI void* hook_RtlVirtualUnwind(uint32_t h, uint64_t b, uint64_t rip, void* f, void* c, void** d, uint64_t* fr, void* cp) { return NULL; }

MS_ABI uint32_t hook_GetACP(void) { return 65001; }
MS_ABI uint32_t hook_GetOEMCP(void) { return 65001; }
MS_ABI uint32_t hook_GetConsoleCP(void) { return 65001; }
MS_ABI uint32_t hook_GetConsoleOutputCP(void) { return 65001; }
MS_ABI int hook_SetConsoleCP(uint32_t cp) { return 1; }
MS_ABI int hook_SetConsoleOutputCP(uint32_t cp) { return 1; }

MS_ABI int hook_MultiByteToWideChar(uint32_t cp, uint32_t flags, const char* mb, int mb_len, wchar_t* wc, int wc_len) {
    if (!mb) return 0;
    if (mb_len < 0) mb_len = strlen(mb) + 1;
    if (wc_len == 0) return mb_len;
    for (int i = 0; i < mb_len && i < wc_len; i++) wc[i] = (wchar_t)(unsigned char)mb[i];
    return (mb_len < wc_len) ? mb_len : wc_len;
}

MS_ABI int hook_WideCharToMultiByte(uint32_t cp, uint32_t flags, const wchar_t* wc, int wc_len, char* mb, int mb_len, const char* def, int* used) {
    if (!wc) return 0;
    if (wc_len < 0) wc_len = wcslen(wc) + 1;
    if (mb_len == 0) return wc_len;
    for (int i = 0; i < wc_len && i < mb_len; i++) mb[i] = (char)wc[i];
    return (wc_len < mb_len) ? wc_len : mb_len;
}

MS_ABI wchar_t* hook_GetEnvironmentStringsW(void) {
    static wchar_t env[] = L"PATH=/bin\0ALLUSERSPROFILE=C:\\ProgramData\0\0";
    return env;
}
MS_ABI int hook_FreeEnvironmentStringsW(void* p) { return 1; }

typedef struct {
    uint32_t cb;
    void* lpReserved; void* lpDesktop; void* lpTitle;
    uint32_t dwX, dwY, dwXSize, dwYSize, dwXCountChars, dwYCountChars, dwFillAttribute, dwFlags;
    uint16_t wShowWindow, cbReserved2;
    void* lpReserved2;
    void* hStdInput; void* hStdOutput; void* hStdError;
} STARTUPINFOW;

MS_ABI void hook_GetStartupInfoW(STARTUPINFOW* si) {
    if (!si) return;
    memset(si, 0, sizeof(STARTUPINFOW));
    si->cb = sizeof(STARTUPINFOW);
    si->hStdInput = (void*)0;
    si->hStdOutput = (void*)1;
    si->hStdError = (void*)2;
}

MS_ABI int hook_BCryptGenRandom(void* h, uint8_t* buf, uint32_t count, uint32_t flags) {
    if (buf) {
        for (uint32_t i = 0; i < count; i++) buf[i] = (uint8_t)(i ^ 0x5A);
    }
    return 0;
}

MS_ABI uint32_t hook_FormatMessageW(uint32_t dwFlags, const void* lpSource, uint32_t dwMessageId, uint32_t dwLanguageId, wchar_t* lpBuffer, uint32_t nSize, void* Arguments) {
    if (lpBuffer && nSize > 0) lpBuffer[0] = L'\0';
    return 0;
}

MS_ABI void* hook_LocalFree(void* hMem) { return NULL; }
MS_ABI int hook_SetThreadStackGuarantee(uint32_t* size) { return 1; }
MS_ABI int hook_SetThreadDescription(void* h, const wchar_t* desc) { return 1; }

MS_ABI void* hook_GetProcAddress(void* h, const char* name);
MS_ABI void* hook_LoadLibraryA(const char* n) { return (void*)0x20000; }
MS_ABI void* hook_LoadLibraryW(const wchar_t* n) { return (void*)0x20000; }
MS_ABI void* hook_LoadLibraryExW(const wchar_t* n, void* f, uint32_t fl) { return (void*)0x20000; }
MS_ABI int hook_FreeLibrary(void* h) { return 1; }

typedef struct { const char* name; void* func; } Sym;
static Sym g_symbols[] = {
    {"GetModuleHandleA", hook_GetModuleHandleA},
    {"GetModuleHandleW", hook_GetModuleHandleW},
    {"GetModuleHandleExW", hook_GetModuleHandleExW},
    {"GetCommandLineA", hook_GetCommandLineA},
    {"GetCommandLineW", hook_GetCommandLineW},
    {"GetStdHandle", hook_GetStdHandle},
    {"GetProcessHeap", hook_GetProcessHeap},
    {"HeapAlloc", hook_HeapAlloc},
    {"HeapFree", hook_HeapFree},
    {"HeapReAlloc", hook_HeapReAlloc},
    {"WriteFile", hook_WriteFile},
    {"WriteConsoleA", hook_WriteConsoleA},
    {"WriteConsoleW", hook_WriteConsoleW},
    {"ExitProcess", hook_ExitProcess},
    {"GetFileType", hook_GetFileType},
    {"GetConsoleMode", hook_GetConsoleMode},
    {"SetConsoleMode", hook_SetConsoleMode},
    {"GetLastError", hook_GetLastError},
    {"SetLastError", hook_SetLastError},
    {"SetUnhandledExceptionFilter", hook_SetUnhandledExceptionFilter},
    {"AddVectoredExceptionHandler", hook_AddVectoredExceptionHandler},
    {"GetCurrentProcessId", hook_GetCurrentProcessId},
    {"GetCurrentThreadId", hook_GetCurrentThreadId},
    {"GetSystemTimeAsFileTime", hook_GetSystemTimeAsFileTime},
    {"GetSystemTimePreciseAsFileTime", hook_GetSystemTimePreciseAsFileTime},
    {"QueryPerformanceCounter", hook_QueryPerformanceCounter},
    {"QueryPerformanceFrequency", hook_QueryPerformanceFrequency},
    {"GetSystemInfo", hook_GetSystemInfo},
    {"VirtualAlloc", hook_VirtualAlloc},
    {"VirtualFree", hook_VirtualFree},
    {"TlsAlloc", hook_TlsAlloc},
    {"TlsGetValue", hook_TlsGetValue},
    {"TlsSetValue", hook_TlsSetValue},
    {"TlsFree", hook_TlsFree},
    {"AcquireSRWLockExclusive", hook_AcquireSRWLockExclusive},
    {"ReleaseSRWLockExclusive", hook_ReleaseSRWLockExclusive},
    {"AcquireSRWLockShared", hook_AcquireSRWLockShared},
    {"ReleaseSRWLockShared", hook_ReleaseSRWLockShared},
    {"InitializeSRWLock", hook_InitializeSRWLock},
    {"SleepConditionVariableSRW", hook_SleepConditionVariableSRW},
    {"WakeAllConditionVariable", hook_WakeAllConditionVariable},
    {"WakeConditionVariable", hook_WakeConditionVariable},
    {"InitializeCriticalSection", hook_InitializeCriticalSection},
    {"InitializeCriticalSectionAndSpinCount", hook_InitializeCriticalSectionAndSpinCount},
    {"EnterCriticalSection", hook_EnterCriticalSection},
    {"LeaveCriticalSection", hook_LeaveCriticalSection},
    {"DeleteCriticalSection", hook_DeleteCriticalSection},
    {"GetACP", hook_GetACP},
    {"GetOEMCP", hook_GetOEMCP},
    {"GetConsoleCP", hook_GetConsoleCP},
    {"GetConsoleOutputCP", hook_GetConsoleOutputCP},
    {"SetConsoleCP", hook_SetConsoleCP},
    {"SetConsoleOutputCP", hook_SetConsoleOutputCP},
    {"MultiByteToWideChar", hook_MultiByteToWideChar},
    {"WideCharToMultiByte", hook_WideCharToMultiByte},
    {"GetEnvironmentStringsW", hook_GetEnvironmentStringsW},
    {"FreeEnvironmentStringsW", hook_FreeEnvironmentStringsW},
    {"GetStartupInfoW", hook_GetStartupInfoW},
    {"BCryptGenRandom", hook_BCryptGenRandom},
    {"FormatMessageW", hook_FormatMessageW},
    {"LocalFree", hook_LocalFree},
    {"SetThreadStackGuarantee", hook_SetThreadStackGuarantee},
    {"SetThreadDescription", hook_SetThreadDescription},
    {"RtlCaptureContext", hook_RtlCaptureContext},
    {"RtlLookupFunctionEntry", hook_RtlLookupFunctionEntry},
    {"RtlVirtualUnwind", hook_RtlVirtualUnwind},
    {"GetProcAddress", hook_GetProcAddress},
    {"LoadLibraryA", hook_LoadLibraryA},
    {"LoadLibraryW", hook_LoadLibraryW},
    {"LoadLibraryExW", hook_LoadLibraryExW},
    {"FreeLibrary", hook_FreeLibrary},
    {NULL, NULL}
};

void* resolve_sym(const char* name) {
    if (!name || (uintptr_t)name < 0x10000) return NULL;
    for (int i = 0; g_symbols[i].name; i++) {
        if (strcmp(g_symbols[i].name, name) == 0) return g_symbols[i].func;
    }
    return NULL;
}

MS_ABI void* hook_GetProcAddress(void* h, const char* name) {
    void* s = resolve_sym(name);
    return s; // Возвращаем NULL для неизвестных API — штатно для Windows!
}

int main() {
    struct sigaction sa;
    memset(&sa, 0, sizeof(sa));
    sa.sa_sigaction = crash_handler;
    sa.sa_flags = SA_SIGINFO;
    sigaction(SIGSEGV, &sa, NULL);
    sigaction(SIGILL, &sa, NULL);
    sigaction(SIGBUS, &sa, NULL);

    FILE* f = fopen("target_app.exe", "rb");
    if (!f) { printf("[-] target_app.exe not found!\n"); return 1; }
    fseek(f, 0, SEEK_END);
    size_t sz = ftell(f);
    fseek(f, 0, SEEK_SET);
    uint8_t* pe = malloc(sz);
    if (!pe || fread(pe, 1, sz, f) != sz) return 1;
    fclose(f);

    uint32_t elfanew = *(uint32_t*)(pe + 0x3C);
    uint16_t nsec = *(uint16_t*)(pe + elfanew + 0x6);
    uint8_t* opt = pe + elfanew + 0x18;
    uint64_t pref = *(uint64_t*)(opt + 0x18);
    uint32_t entry_rva = *(uint32_t*)(opt + 0x10);
    uint32_t img_sz = *(uint32_t*)(opt + 0x38);
    uint32_t hdr_sz = *(uint32_t*)(opt + 0x3C);
    uint8_t* dirs = opt + 0x70;
    uint32_t imp_rva = *(uint32_t*)(dirs + 0x8);
    uint32_t rel_rva = *(uint32_t*)(dirs + 0x28);
    uint32_t rel_sz  = *(uint32_t*)(dirs + 0x2C);
    uint32_t tls_rva = *(uint32_t*)(dirs + 0x48);

    g_base = (uint8_t*)mmap((void*)pref, img_sz, PROT_READ | PROT_WRITE | PROT_EXEC, MAP_PRIVATE | MAP_ANONYMOUS, -1, 0);
    if (g_base == MAP_FAILED) g_base = (uint8_t*)mmap(NULL, img_sz, PROT_READ | PROT_WRITE | PROT_EXEC, MAP_PRIVATE | MAP_ANONYMOUS, -1, 0);

    memcpy(g_base, pe, hdr_sz);
    uint8_t* sec_hdr = opt + *(uint16_t*)(pe + elfanew + 0x14);
    for (int i = 0; i < nsec; i++) {
        uint8_t* s = sec_hdr + i * 40;
        uint32_t va = *(uint32_t*)(s + 12), vs = *(uint32_t*)(s + 8), rs = *(uint32_t*)(s + 16), rp = *(uint32_t*)(s + 20);
        if (rs > 0) memcpy(g_base + va, pe + rp, rs);
        if (vs > rs) memset(g_base + va + rs, 0, vs - rs);
    }

    // Релокации
    int64_t delta = (int64_t)g_base - (int64_t)pref;
    if (rel_rva && delta != 0) {
        uint32_t off = 0;
        while (off < rel_sz) {
            uint32_t pg = *(uint32_t*)(pe + rel_rva + off), blk = *(uint32_t*)(pe + rel_rva + off + 4);
            if (!blk) break;
            int cnt = (blk - 8) / 2;
            uint16_t* ents = (uint16_t*)(pe + rel_rva + off + 8);
            for (int j = 0; j < cnt; j++) {
                if ((ents[j] >> 12) == 10) {
                    uint64_t* ptr = (uint64_t*)(g_base + pg + (ents[j] & 0xFFF));
                    *ptr += delta;
                }
            }
            off += blk;
        }
    }

    // IAT
    if (imp_rva) {
        uint8_t* desc = g_base + imp_rva;
        while (1) {
            uint32_t orig = *(uint32_t*)desc, ft = *(uint32_t*)(desc + 16);
            if (!orig && !ft) break;
            uint32_t th_rva = orig ? orig : ft;
            uint64_t* thunk = (uint64_t*)(g_base + th_rva);
            uint64_t* iat = (uint64_t*)(g_base + ft);
            while (*thunk) {
                if (!(*thunk & (1ULL << 63))) {
                    char* name = (char*)(g_base + *thunk + 2);
                    void* sym = resolve_sym(name);
                    *iat = (uint64_t)sym;
                }
                thunk++; iat++;
            }
            desc += 20;
        }
    }

    // TLS
    if (tls_rva) {
        uint64_t* tls_dir_raw = (uint64_t*)(g_base + tls_rva);
        uint64_t raw_start = tls_dir_raw[0];
        uint64_t raw_end = tls_dir_raw[1];
        uint64_t addr_idx = tls_dir_raw[2];
        size_t raw_sz = (raw_end > raw_start) ? (raw_end - raw_start) : 0;

        void* tls_data = hook_HeapAlloc(NULL, 0, raw_sz + 512);
        if (raw_sz > 0) memcpy(tls_data, (void*)raw_start, raw_sz);
        g_tls_ptrs[0] = tls_data;

        if (addr_idx) {
            uint32_t* pidx = (uint32_t*)addr_idx;
            *pidx = 0;
        }
    }

    memset(&g_proc_params, 0, sizeof(g_proc_params));
    g_proc_params.CommandLine.Length = wcslen(g_cmdline) * 2;
    g_proc_params.CommandLine.MaximumLength = sizeof(g_cmdline);
    g_proc_params.CommandLine.Buffer = g_cmdline;

    g_proc_params.ImagePathName.Length = wcslen(g_imagepath) * 2;
    g_proc_params.ImagePathName.MaximumLength = sizeof(g_imagepath);
    g_proc_params.ImagePathName.Buffer = g_imagepath;

    memset(&g_peb, 0, sizeof(g_peb));
    g_peb.ProcessParameters = &g_proc_params;
    g_peb.ProcessHeap = (void*)g_heap_buf;
    g_peb.FastPebLock = &g_fast_peb_lock;

    memset(&g_teb, 0, sizeof(g_teb));
    g_teb.Self = &g_teb;
    g_teb.TlsArray = g_tls_ptrs;
    g_teb.ProcessEnvironmentBlock = &g_peb;
    syscall(SYS_arch_prctl, ARCH_SET_GS, &g_teb);

    printf("[+] Executing Windows PE Binary (ripgrep) at %p...\n\n", g_base);
    fflush(stdout);

    void* entry_addr = (void*)(g_base + entry_rva);

    if (setjmp(g_exit_env) == 0) {
        __asm__ __volatile__(
            "sub $40, %%rsp\n\t"
            "call *%0\n\t"
            "add $40, %%rsp\n\t"
            :
            : "r"(entry_addr)
            : "rax", "rcx", "rdx", "r8", "r9", "r10", "r11", "memory"
        );
    }

    printf("\n[+] Binary execution finished cleanly with ExitCode: %d\n", g_exit_code);
    return 0;
}

Overwriting rg_loader.c


In [50]:
%%bash
gcc -O2 rg_loader.c -o rg_loader
./rg_loader

[+] Executing Windows PE Binary (ripgrep) at 0x140000000...


[FATAL] Signal 11 (Segmentation fault) at (nil)
        RIP: 0x5b75f16f46e0 (PE Base: 0x140000000), RSP: 0x7ffcdbd2f4d8
        RAX: 0x0, RCX: 0x20000, RDX: 0x1404818a8


CalledProcessError: Command 'b'gcc -O2 rg_loader.c -o rg_loader\n./rg_loader\n'' returned non-zero exit status 139.